# 07 — Thí nghiệm khẳng định (đăng ký trước)

**Đề tài:** Trôi phiên bản tri thức trong trợ lý CTI dùng LLM/RAG

Notebook 05 và 06 là thí nghiệm **khám phá**. Chúng dùng hai mô hình (Qwen3-8B, Llama-3.1-8B) và một lần chuyển phiên bản (v18.1 → v19.2). Notebook này **kiểm định lại** các phát hiện chính theo kế hoạch **định trước**: giả thuyết, chỉ số, phép kiểm định và quy tắc kết luận đều được viết ra **trước khi chạy**. Kế hoạch có hai nhánh.

| Nhánh | Lần chuyển | Mô hình | Vì sao |
|---|---|---|---|
| **A** | v18.1 → v19.2 | 3 mô hình **mới**, khác họ: Gemma-2-9B, Phi-4-mini, Mistral-7B-v0.3 | Kiểm tra phát hiện có đúng với mô hình chưa từng dùng để khám phá không |
| **B** | v6.3 → v7.2 | cả 5 mô hình | Kiểm tra trên một lần chuyển **khác**, nơi kiến thức sẵn có của mô hình **nghiêng về bản mới** |

**Vì sao chọn v6.3 → v7.2.** Thầy đã đếm thay đổi giữa mọi bản lớn từ v6.3 tới v19.2. Ngoài v18.1 → v19.2, **chỉ có** v6.3 → v7.2 (năm 2020, đưa sub-technique vào) có đủ thay đổi để thí nghiệm:

- 129 mã bị thu hồi;
- 10 kỹ thuật đổi tactic.

Các lần chuyển khác từ v7.2 tới v18.1 có **0–6** kỹ thuật đổi tactic và **0–2** mã bị thu hồi mỗi lần. Ngoài ra, cả 5 mô hình đều được huấn luyện **sau** năm 2020, nên kiến thức sẵn có của chúng nghiêng về **bản mới** (v7 trở đi). Điều này ngược hẳn với v18.1 → v19.2, nơi kiến thức sẵn có nghiêng về bản cũ. Nhờ vậy nhánh B kiểm tra được giả thuyết "thiên vị kiến thức sẵn có".

**Cách chạy:** chạy notebook **hai lần**. Lần đầu đặt `TRANSITION = "18.1-19.2"`, lần sau đặt `TRANSITION = "6.3-7.2"`. Kết quả ghi vào `results/conf-<lần chuyển>/`. Mục 9 so sánh hai lần chuyển, và chỉ chạy được khi đã có kết quả của cả hai.

## Kế hoạch định trước

### Điều kiện ngữ cảnh

Mười hai điều kiện, mỗi điều kiện gồm 4 đoạn, dựng **giống hệt** notebook 05 và 06:

| Điều kiện | Ngữ cảnh | Nhãn trên đoạn |
|---|---|---|
| `new`, `old` | chỉ N / chỉ O | không |
| `old_new`, `new_old` | O,N / N,O | không |
| `tag_old_new`, `tag_new_old` | O,N / N,O | đầy đủ: `vX (superseded)` / `vY (current release)` |
| `vtag_old_new`, `vtag_new_old` | O,N / N,O | chỉ số phiên bản |
| `swap_old_new`, `swap_new_old` | O,N / N,O | tráo: đoạn cũ ghi *current*, đoạn mới ghi *superseded* |
| `bridge_old_new`, `bridge_new_old` | O,N,B / N,O,B | đầy đủ, cộng ghi chú cầu nối B cho mã bị thu hồi |

**Cách hỏi.** Mọi điều kiện dùng cách hỏi `ver` (câu hỏi có số phiên bản mới). Riêng `vtag_*` hỏi thêm theo cách `nover` (bỏ số phiên bản), vì C4 cần.

### Giả thuyết khẳng định

Chỉ số là `drift_ext` (tỉ lệ trả lời theo bản cũ, chỉ xét mã), tính trên nhóm `changed`. Mọi so sánh đều ghép cặp trên cùng câu hỏi. "Gộp" nghĩa là lấy chung cả hai thứ tự O/N.

| | Giả thuyết | So sánh (dự đoán > 0) | Nhánh |
|---|---|---|---|
| **C1** | Sự có mặt của đoạn cũ gây lỗi thời | drift(`new_old`) − drift(`new`) | A (B chỉ báo cáo, xem dưới) |
| **C2** | Nhãn phiên bản giảm lỗi thời | drift(`old_new`,`new_old`) − drift(`tag_*`), gộp | A, B |
| **C3** | Chữ *current/superseded* trong nhãn cần thiết | drift(`vtag_*`) − drift(`tag_*`), gộp | A, B |
| **C4** | Nhãn chỉ có số phiên bản hoạt động nhờ **khớp chuỗi** với câu hỏi | drift(`vtag_*`, nover) − drift(`vtag_*`, ver), gộp, **không gồm D4** | A, B |
| **C5** | Siêu dữ liệu lái được câu trả lời (nhãn tráo) | drift(`swap_*`) − drift(`tag_*`), gộp | A, B |
| **C6** | Ghi chú cầu nối giúp trả lời đúng với mã bị thu hồi | **acc**(`bridge_*`) − acc(`tag_*`), gộp, **chỉ D1 có mã bị thu hồi** | A, B |
| **C7** | Thiên vị kiến thức sẵn có | drift(không nhãn, A) − drift(không nhãn, B), cùng mô hình | so sánh A với B |

Vài ghi chú cho từng giả thuyết:

- **C1 ở nhánh B.** Theo giả thuyết "thiên vị kiến thức sẵn có", ở v6.3 → v7.2 mô hình nghiêng về bản mới, nên hiệu ứng có thể nhỏ hoặc đổi chiều. Vì vậy C1 ở nhánh B chỉ được báo cáo với khoảng tin cậy hai phía, không nằm trong họ kiểm định.
- **C4 không gồm D4.** Câu D4 ở dạng `nover` phải thêm chữ *current*, tức là đổi hai thứ cùng lúc.
- **C6 chỉ xét D1 có mã bị thu hồi.** Với D4, ghi chú cầu nối nói thẳng đáp án. Ở D1, mô hình vẫn phải ghép ghi chú với đoạn N để suy ra tactic.
- **C1 không chỉ thêm đoạn cũ.** So với `new` (N,D,D,D), điều kiện `new_old` (N,O,D,D) có đoạn cũ thay vào chỗ một đoạn nhiễu. Cách dựng này kế thừa từ notebook 05.
- **C7 so sánh giữa hai tập mục khác nhau.** Hai nhánh có thành phần mục rất khác: nhánh B chủ yếu là mã bị thu hồi. Vì vậy C7 dùng **ước lượng phân tầng** theo ba tầng có ở cả hai nhánh:
  - D1 còn hiệu lực;
  - D1 có mã bị thu hồi;
  - D4 có mã bị thu hồi.

  Hiệu của mỗi tầng được gộp với trọng số w = nA·nB / (nA + nB), trong đó n là số câu của tầng đó ở mỗi nhánh.

**Nhánh B kiểm định gì.** Phần lớn câu `changed` ở nhánh B là **mã bị thu hồi**: 191/201 câu, trong đó D4 chiếm 129. Vì vậy nhánh B chủ yếu kiểm tra lỗi thời do **thu hồi mã**, không phải do **đổi tactic** như nhánh A.

**Nhánh B và chữ "current".** Mô hình biết v7.2 (năm 2020) không phải bản hiện hành, nên nhãn "v7.2 (current release)" và câu hỏi "current release (v7.2)" có thể kém thuyết phục với mô hình. Đây là bối cảnh "giả lập hiện tại". Kết quả nhánh B được đọc theo quy tắc dưới đây, và **không** dùng để bác bỏ kết luận của nhánh A.

### Kiểm định và quy tắc kết luận

**Kiểm định cho từng mô hình:**

- **Giá trị p một phía** lấy từ **phép đổi dấu theo cụm** (cluster sign-flip, 10.000 lần), làm trên hiệu ghép cặp của từng cụm dòng dõi. Phép này đúng cả khi có ít cụm (C6 ở nhánh A chỉ có khoảng 13 cụm), còn bootstrap thì cho p quá nhỏ trong trường hợp đó.
- **Khoảng tin cậy 95%** để ước lượng độ lớn hiệu dùng bootstrap theo cụm (B = 10.000).
- **Riêng C7** so sánh hai tập mục khác nhau, nên giá trị p lấy từ bootstrap độc lập: p = (1 + số lần hiệu ≤ 0) / (B + 1).
- Hiệu chỉnh **Holm** cho họ kiểm định của mỗi (mô hình × nhánh):
  - nhánh A: C1–C6;
  - nhánh B: C2–C6.
- Một giả thuyết **đạt** ở một mô hình khi p sau Holm < 0,05.

**Quy tắc kết luận:**

- **Nhánh A:** giả thuyết được **khẳng định** nếu đạt ở **ít nhất 2/3** mô hình mới và **không** mô hình nào có khoảng tin cậy 95% nằm hẳn dưới 0.
- **Nhánh B:** cần đạt ở **ít nhất 4/5** mô hình, cùng điều kiện "không mô hình nào ngược chiều".
- **Quy tắc sàn cho C2:** nếu lỗi thời **không nhãn** (gộp hai thứ tự) của một mô hình dưới **0,15**, nhãn không còn chỗ để giảm lỗi thời. Khi đó C2 của mô hình đó được ghi **"sàn – không đánh giá được"**:
  - C2 của mô hình đó bị loại khỏi họ Holm;
  - mẫu số của quy tắc kết luận giảm tương ứng, và ngưỡng đạt là **80% số mô hình còn lại, làm tròn lên**;
  - nếu còn dưới 3 mô hình đánh giá được thì C2 của nhánh đó là "không đánh giá được".
- **Mô hình lỗi kỹ thuật:** một mô hình không chạy được, hoặc có tỉ lệ JSON hợp lệ dưới 0,95.
  - Lỗi môi trường (tải mô hình, hết bộ nhớ…) phải được **sửa rồi chạy lại**. Đó không phải loại bỏ theo kết quả.
  - **Thang cấu hình môi trường dự phòng (định trước).** Nếu một mô hình chạy lỗi (khi chạy thử hoặc khi chạy chính), notebook tự thử lần lượt các cấu hình dự phòng sau. Chúng chỉ đổi cách tính trên GPU, không đổi prompt, cách giải mã (greedy, JSON) hay cách chấm:
    1. **cuBLASLt:** dùng thư viện nhân ma trận cuBLASLt thay cho cuBLAS (`TORCH_BLAS_PREFER_CUBLASLT=1`);
    2. **gọn bộ nhớ:** chạy không biên dịch (`enforce_eager`), lô nhỏ (`max_num_seqs = 32`, `max_num_batched_tokens = 2048`), `max_model_len = 4096` (prompt dài nhất dưới khoảng 2.000 token; nếu vượt, notebook dừng và báo) và `gpu_mem = 0,80`. Lượt chạy đo bộ nhớ lúc khởi động của vLLM nhẹ đi nhiều: ít token hơn mỗi bước, và lớp đầu ra cùng bộ lấy mẫu chỉ chạy với tối đa 32 câu một lúc;
    3. cả hai cách trên.

    Quy tắc:
    - Mỗi mô hình dùng cấu hình **đầu tiên** chạy được, và chỉ đi lên thang, không quay lại, kể cả khi chạy lại notebook (notebook đọc `env_rung.json` của lần chạy trước). Mô hình chạy được ở cấu hình mặc định thì giữ cấu hình mặc định.
    - Nếu chạy chính lỗi ở cấu hình đã qua chạy thử, notebook thử tiếp các cấu hình cao hơn cho phần chạy chính.
    - Ở nhánh B, mỗi mô hình **bắt đầu** từ cấu hình nó đã dùng ở nhánh A, để C7 so sánh trên cùng cấu hình. Nếu ở nhánh B mô hình phải lên cấu hình cao hơn, C7 vẫn được kiểm định và bảng C7 ghi rõ cấu hình của hai nhánh.
    - Lỗi do prompt quá dài là lỗi của notebook, không phải lỗi môi trường: notebook dừng hẳn.
    - Chỉ quay về cấu hình mặc định khi môi trường đã được **sửa thật** (ví dụ cài lại thư viện) và thầy đồng ý: khi đó xoá `env_rung.json` của nhánh đó rồi chạy lại từ ô chạy thử. Việc này được ghi vào nhật ký phiên bản.
    - Cấu hình dùng cho từng mô hình, cùng mọi lần thử, được ghi vào `env_rung.json` và manifest. Bài báo sẽ báo cáo cấu hình của từng mô hình.
    - Các cấu hình có thể làm câu trả lời lệch rất nhẹ (thứ tự cộng số thực khác nhau), giống như chạy trên GPU khác.

    Thang này được thêm ở bản 1.1 và chỉnh ở bản 1.2, sau khi Gemma-2 không khởi động được ở bản 1.0. Nguyên nhân là GPU hết bộ nhớ trong lượt chạy đo bộ nhớ của vLLM; cuBLAS không xin được vùng nhớ làm việc nên báo `CUBLAS_STATUS_NOT_SUPPORTED`. Thang chỉ đổi cách tính trên GPU, không phụ thuộc vào kết quả của mô hình nào.
  - Nếu sửa rồi vẫn không chạy được, mô hình **giữ trong mẫu số** và bị tính là **không đạt** mọi giả thuyết. Mô hình đó không được tính là "ngược chiều", và không được thay bằng mô hình khác.
  - Notebook ghi danh sách mô hình lỗi vào `tech_fail.json` và vẫn chạy tiếp các mô hình còn lại. Sau khi sửa môi trường, chạy lại **từ ô chạy thử** (mục 5); các mô hình đã chạy xong sẽ tự được bỏ qua.
- **Mô hình ở sàn** (với C2) không được tính là "ngược chiều".
- **C7:** kiểm định cho từng mô hình, hiệu chỉnh Holm trên 5 mô hình. C7 được khẳng định nếu đạt ở **ít nhất 4/5** mô hình và không mô hình nào ngược chiều.
- **Cách ghép hai nhánh vào bài:**
  - Kết luận "đúng với nhiều họ mô hình" dựa trên **nhánh A**.
  - Kết luận "đúng với lần chuyển phiên bản khác" chỉ được viết khi **nhánh B** cũng khẳng định giả thuyết đó. Nếu không, giả thuyết được ghi là **riêng cho v18.1 → v19.2**.

**Phân tích độ nhạy:** lặp lại tất cả với chỉ số thay thế `stale13` của bộ chấm v1.3. `stale13` tính cả tên cũ, và không tính các câu ghép mã–tên lệch, nên **không** hẳn chặt hơn. Kết luận được báo cáo cả khi hai cách chấm khác nhau.

**Những gì KHÔNG được làm sau khi xem kết quả:**

- không đổi chỉ số, điều kiện, mô hình, tập mục hay quy tắc kết luận;
- không bỏ mô hình nào vì kết quả "xấu".

Mọi phân tích thêm đều phải ghi là **khám phá**.

**Nên đăng ký kế hoạch này** (ví dụ trên OSF) trước khi chạy. Kèm theo mã băm của file dựng `build_07.py`, hoặc mã băm của nội dung các ô code; đừng dùng mã băm của file .ipynb, vì file đó đổi mỗi khi lưu.

### Nhật ký phiên bản

| Phiên bản | Ngày | Thay đổi |
|---|---|---|
| 1.0 | 13/10/2026 | Bản đầu tiên |
| 1.1 | 13/10/2026 | Ở lần chạy bản 1.0, Gemma-2 không khởi động được (`CUBLAS_STATUS_NOT_SUPPORTED`). Lần chạy đó đã đi hết notebook: Phi-4-mini và Mistral chạy chính xong, bảng kiểm định nhánh A đã được tính, Gemma-2 bị tính là lỗi kỹ thuật. Thêm thang cấu hình môi trường dự phòng; module suy luận bản 2 `ctidrift_infer_v2.py` do notebook này ghi ra (thêm tham số máy vLLM, ghi cấu hình BLAS vào khoá chạy và meta). |
| 1.2 | 13/10/2026 | Log đầy đủ của Gemma-2 cho thấy nguyên nhân thật là **hết bộ nhớ GPU** (còn trống 11 MB) trong lượt chạy đo bộ nhớ của vLLM; cuBLAS không xin được vùng nhớ làm việc nên báo `CUBLAS_STATUS_NOT_SUPPORTED`. Cấu hình "gọn bộ nhớ" giảm `max_num_batched_tokens` từ 4096 xuống 2048. |
| 1.3 | 14/10/2026 | Sửa lỗi bộ lọc đoạn nhiễu, phát hiện khi chạy nhánh B (trước khi có prompt hay câu trả lời nào của nhánh B). Bộ lọc chỉ loại tên tactic đứng thành từ riêng, còn phép kiểm tra rò rỉ loại cả khi tên nằm trong từ khác. Đoạn `M1042#2` chứa "ApplePersistence", qua được bộ lọc nhưng bị phép kiểm tra chặn. Nay bộ lọc dùng đúng tiêu chí của phép kiểm tra. Nhánh A không đổi: các đoạn nhiễu đã chọn đều qua phép kiểm tra, nên tiêu chí chặt hơn không làm đổi lựa chọn; phép so khớp từng ký tự với prompt notebook 05/06 vẫn được giữ. |

## 1. Cấu hình

**Em chỉ cần đổi `TRANSITION`.** Mọi tham số khác đã cố định theo kế hoạch.

Ba mô hình mới được ghim theo **commit** trên Hugging Face, đã kiểm tra ngày 13/10/2026:

- **Gemma-2** và **Mistral-7B-v0.3**: khuôn hội thoại của hai mô hình này **không nhận** vai trò `system` (cả hai báo lỗi khi gặp `system`). Vì vậy lời dặn hệ thống được **ghép vào đầu lượt người dùng**, với nội dung giữ nguyên. Ba mô hình còn lại dùng vai trò `system` như notebook 04–06.
- **Phi-4-mini** chạy ở bf16 (khoảng 7,7 GB), với `max_model_len = 4096`. Giới hạn này không vượt quá độ dài gốc của mô hình, nên vLLM không dùng hệ số RoPE dành cho văn bản dài. Prompt dài nhất chỉ khoảng 1.900 token. Các mô hình kia dùng AWQ 4-bit và `max_model_len = 6144` như trước.

**Module suy luận.** Notebook này tự ghi `ctidrift_infer_v2.py` (mục 5) và **không** sửa `ctidrift_infer.py` của notebook 04–06. Bản 2 giống bản 1, chỉ thêm tham số máy vLLM cho thang cấu hình dự phòng (xem "Mô hình lỗi kỹ thuật" ở phần kế hoạch).

Lần đầu chạy, mỗi mô hình mới được tải về `/data/hf`: Gemma-2 và Mistral khoảng 5–6 GB, Phi-4-mini khoảng 7,7 GB.

**Không** chạy lại notebook 02 cho v18.1/v19.2. Danh mục CWE và CVE tải về có thể đã đổi, làm đổi đoạn nhiễu, và phép kiểm tra trùng prompt với notebook 05/06 sẽ báo lỗi. Chỉ cần chạy notebook 02 cho **v6.3 và v7.2**.

In [36]:
TRANSITION = "6.3-7.2"        # ĐỔI Ở ĐÂY: "18.1-19.2" (nhánh A) hoặc "6.3-7.2" (nhánh B)

from pathlib import Path
import os, sys, json, re, math, time, hashlib, subprocess, collections, datetime, zipfile, signal, random
import numpy as np
import pandas as pd

V_OLD, V_NEW = TRANSITION.split("-")
ROOT    = Path.cwd()
DRIFT   = ROOT / "benchmark" / f"drift-{V_OLD}-{V_NEW}"
OUT     = ROOT / "results" / f"conf-{V_OLD}-{V_NEW}"
PROMPTS, PROMPTS_MS = OUT / "prompts", OUT / "prompts_mergedsys"
SMOKE_P, SMOKE_P_MS = OUT / "prompts_smoke", OUT / "prompts_smoke_mergedsys"
RAW, LOGS = OUT / "raw", OUT / "logs"
for p in (OUT, PROMPTS, PROMPTS_MS, SMOKE_P, SMOKE_P_MS, RAW, LOGS):
    p.mkdir(parents=True, exist_ok=True)

HF_HOME         = "/data/hf"
VLLM_CACHE_ROOT = "/data/vllm-cache"
ALL_MODELS = [
    dict(tag="qwen3-8b", repo="Qwen/Qwen3-8B-AWQ", revision="4da05a8edb55c6046cce958586c33b61da07bb79",
         released="2025-05", chat_template_kwargs={"enable_thinking": False}, merge_system=False, explored=True),
    dict(tag="llama3.1-8b", repo="hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4",
         revision="db1f81ad4b8c7e39777509fac66c652eb0a52f91", released="2024-07",
         chat_template_kwargs={}, merge_system=False, explored=True),
    dict(tag="gemma2-9b", repo="hugging-quants/gemma-2-9b-it-AWQ-INT4",
         revision="6e62725da8e92309167814dad7aacc0ed8cb2484", released="2024-06",
         chat_template_kwargs={}, merge_system=True, explored=False),
    dict(tag="phi4-mini", repo="microsoft/Phi-4-mini-instruct",
         revision="cfbefacb99257ffa30c83adab238a50856ac3083", released="2025-02",
         chat_template_kwargs={}, merge_system=False, explored=False,
         max_model_len=4096),   # ≤ original_max_position_embeddings: tránh vLLM dùng hệ số RoPE dài cho câu ngắn
    dict(tag="mistral7b-v0.3", repo="solidrust/Mistral-7B-Instruct-v0.3-AWQ",
         revision="95b1295ddd1a8673117cdc7bd2a4da2a457bb3f7", released="2024-05",
         chat_template_kwargs={}, merge_system=True, explored=False),
]
STUDY = {
    "18.1-19.2": dict(branch="A", source="pilot_queries.jsonl",
                      templates=["D1_tech_tactic", "D2_subtech_parent", "D4_id_currency"], stable_cap=None,
                      models=[m for m in ALL_MODELS if not m["explored"]], family=["C1", "C2", "C3", "C4", "C5", "C6"],
                      min_pass=2),
    "6.3-7.2":   dict(branch="B", source="queries_all.jsonl",
                      templates=["D1_tech_tactic", "D4_id_currency"], stable_cap=80,
                      models=ALL_MODELS, family=["C2", "C3", "C4", "C5", "C6"], min_pass=4),
}
assert TRANSITION in STUDY, f"TRANSITION phải là một trong {list(STUDY)}"
CFG = STUDY[TRANSITION]
MODELS = CFG["models"]

CONDITIONS = {   # order: thứ tự bằng chứng; tag: kiểu nhãn; swap: tráo nhãn; qforms: cách hỏi
    "new":            dict(order=["N"],           tag=None,   swap=False, qforms=["ver"]),
    "old":            dict(order=["O"],           tag=None,   swap=False, qforms=["ver"]),
    "old_new":        dict(order=["O", "N"],      tag=None,   swap=False, qforms=["ver"]),
    "new_old":        dict(order=["N", "O"],      tag=None,   swap=False, qforms=["ver"]),
    "tag_old_new":    dict(order=["O", "N"],      tag="full", swap=False, qforms=["ver"]),
    "tag_new_old":    dict(order=["N", "O"],      tag="full", swap=False, qforms=["ver"]),
    "vtag_old_new":   dict(order=["O", "N"],      tag="ver",  swap=False, qforms=["ver", "nover"]),
    "vtag_new_old":   dict(order=["N", "O"],      tag="ver",  swap=False, qforms=["ver", "nover"]),
    "swap_old_new":   dict(order=["O", "N"],      tag="full", swap=True,  qforms=["ver"]),
    "swap_new_old":   dict(order=["N", "O"],      tag="full", swap=True,  qforms=["ver"]),
    "bridge_old_new": dict(order=["O", "N", "B"], tag="full", swap=False, qforms=["ver"]),
    "bridge_new_old": dict(order=["N", "O", "B"], tag="full", swap=False, qforms=["ver"]),
}
N_PASSAGES      = 4
MAX_CHUNK_WORDS = 250
MAX_MODEL_LEN   = 6144
MAX_TOKENS      = 160
GPU_MEM         = 0.85
SEED            = 2026
BOOT_B          = 10000
ALPHA           = 0.05
SMOKE_PER_GROUP = 1
CONF_VERSION    = "1.3"

# Thang cấu hình môi trường dự phòng (định trước). Chỉ dùng khi mô hình không chạy được ở cấu hình trước đó.
# gpu_mem / max_model_len = None: giữ giá trị chung (GPU_MEM, max_model_len của mô hình).
LEAN = {"enforce_eager": True, "max_num_seqs": 32, "max_num_batched_tokens": 2048}
LT   = {"TORCH_BLAS_PREFER_CUBLASLT": "1"}
ENV_LADDER = [
    dict(name="mặc định",              env={}, engine={},   gpu_mem=None, max_model_len=None),
    dict(name="cuBLASLt",              env=LT, engine={},   gpu_mem=None, max_model_len=None),
    dict(name="gọn bộ nhớ",            env={}, engine=LEAN, gpu_mem=0.80, max_model_len=4096),
    dict(name="cuBLASLt + gọn bộ nhớ", env=LT, engine=LEAN, gpu_mem=0.80, max_model_len=4096),
]

REQUIRED = {
    "queries":        DRIFT / CFG["source"],
    "items":          DRIFT / "items.jsonl",
    "drift_manifest": DRIFT / "manifest.json",
    "tactic_alias":   DRIFT / "tactic_alias.json",
    "corpus_old":     ROOT / "benchmark" / f"attack-{V_OLD}" / "corpus.jsonl",
    "corpus_new":     ROOT / "benchmark" / f"attack-{V_NEW}" / "corpus.jsonl",
    "scoring_module": ROOT / "ctidrift_scoring.py",
    "scoring_v13":    ROOT / "ctidrift_scoring_v13.py",
}
missing = [f"{k}: {p}" for k, p in REQUIRED.items() if not p.exists()]
assert not missing, ("Thiếu đầu vào. Với 6.3-7.2 cần chạy notebook 02 (v6.3 và v7.2) và notebook 03 bản 1.3 "
                     "(V_OLD, V_NEW = \"6.3\", \"7.2\") trước; bộ chấm v1.3 do notebook 06 ghi ra:\n  " + "\n  ".join(missing))

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

DM = json.loads(REQUIRED["drift_manifest"].read_text())
for k, f in (("queries", CFG["source"]), ("items", "items.jsonl")):
    assert DM["outputs"][f] == sha256(REQUIRED[k]), f"{f} không khớp manifest notebook 03"

ENV = {**os.environ, "HF_HOME": HF_HOME, "HF_HUB_CACHE": f"{HF_HOME}/hub", "VLLM_CACHE_ROOT": VLLM_CACHE_ROOT,
       "VLLM_USE_FLASHINFER_SAMPLER": "0", "TOKENIZERS_PARALLELISM": "false"}
ENV.pop("HF_HUB_OFFLINE", None)
ENV.pop("TORCH_BLAS_PREFER_CUBLASLT", None)     # cấu hình BLAS chỉ do thang dự phòng quyết định
ENV.pop("TORCH_BLAS_PREFER_HIPBLASLT", None)

MANIFEST = {"conf_version": CONF_VERSION, "transition": TRANSITION, "branch": CFG["branch"],
            "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "config": dict(CONDITIONS=CONDITIONS, STUDY={k: {kk: vv for kk, vv in v.items() if kk != "models"}
                                                          for k, v in STUDY.items()},
                           N_PASSAGES=N_PASSAGES, MAX_CHUNK_WORDS=MAX_CHUNK_WORDS, MAX_MODEL_LEN=MAX_MODEL_LEN,
                           MAX_TOKENS=MAX_TOKENS, GPU_MEM=GPU_MEM, SEED=SEED, BOOT_B=BOOT_B, ALPHA=ALPHA,
                           ENV_LADDER=ENV_LADDER),
            "models": MODELS,
            "inputs": {k: {"path": str(p.relative_to(ROOT)), "sha256": sha256(p)} for k, p in REQUIRED.items()}}
print(f"Nhánh {CFG['branch']}: ATT&CK v{V_OLD} → v{V_NEW}; mô hình: {[m['tag'] for m in MODELS]}")
for k, v in MANIFEST["inputs"].items():
    print(f"  {k:15s} {v['sha256'][:12]}…  {v['path']}")

Nhánh B: ATT&CK v6.3 → v7.2; mô hình: ['qwen3-8b', 'llama3.1-8b', 'gemma2-9b', 'phi4-mini', 'mistral7b-v0.3']
  queries         58a6ccb4646c…  benchmark/drift-6.3-7.2/queries_all.jsonl
  items           c4849c308e3a…  benchmark/drift-6.3-7.2/items.jsonl
  drift_manifest  d5ada2826ad4…  benchmark/drift-6.3-7.2/manifest.json
  tactic_alias    e082cfab19e8…  benchmark/drift-6.3-7.2/tactic_alias.json
  corpus_old      127261d010d2…  benchmark/attack-6.3/corpus.jsonl
  corpus_new      1062f267ff29…  benchmark/attack-7.2/corpus.jsonl
  scoring_module  56b3cb0eb0c8…  ctidrift_scoring.py
  scoring_v13     0353476caf82…  ctidrift_scoring_v13.py


## 2. Tập câu hỏi

- **Nhánh A:** dùng **đúng** tập câu hỏi của notebook 05 và 06, tức các câu pilot với cách hỏi `current`, mẫu D1, D2, D4, gồm ba nhóm `changed`, `label`, `stable`.
- **Nhánh B:** dùng **toàn bộ** câu có đáp án đổi (nhóm `changed`) của mẫu D1 và D4. Nhóm đối chứng `stable` lấy ngẫu nhiên 80 câu mỗi mẫu, với hạt giống cố định. Lần chuyển này không có D2 (v6.3 chưa có sub-technique) và không có nhóm `label` (không tactic nào đổi tên).

Câu `nover` được tạo bằng cách bỏ số phiên bản khỏi câu hỏi, như notebook 06.

In [37]:
import importlib, ctidrift_scoring, ctidrift_scoring_v13
importlib.reload(ctidrift_scoring); importlib.reload(ctidrift_scoring_v13)
from ctidrift_scoring import score_drift
from ctidrift_scoring_v13 import score_v13

ALIAS_FILE = json.loads(REQUIRED["tactic_alias"].read_text())
TACTIC_ALIAS = ALIAS_FILE["alias"]
ITEMS = {it["item_id"]: it for it in map(json.loads, open(REQUIRED["items"], encoding="utf-8"))}
CORP = {V_OLD: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_old"], encoding="utf-8"))},
        V_NEW: {c["_id"]: c for c in map(json.loads, open(REQUIRED["corpus_new"], encoding="utf-8"))}}

def group_of(q):                         # giống notebook 05, 06
    if q["answer_changed"]:
        return "changed"
    if q["pair_class"] == "LABEL_CHANGED":
        return "label"
    if q["pair_class"] == "STABLE":
        return "stable"
    return None

def evidence_new(q, it):                 # giống notebook 05, 06
    ev = q["evidence"][V_NEW]
    return ev if ev else [f"{s}#0" for s in it["anchor_new"] if f"{s}#0" in CORP[V_NEW]]

def nover(question):                     # giống notebook 06, tổng quát theo V_NEW
    out = question.replace(f" (v{V_NEW})", "")
    out = re.sub(rf"^As of MITRE ATT&CK Enterprise v{re.escape(V_NEW)},", "In the current MITRE ATT&CK Enterprise release,", out)
    return out

cand = []
for q in map(json.loads, open(REQUIRED["queries"], encoding="utf-8")):
    g = group_of(q)
    if q["direction"] != "current" or q["framing"] != "current" or q["template"] not in CFG["templates"] or g is None:
        continue
    cand.append({**q, "group": g})
if CFG["stable_cap"]:
    keep = set()
    for tpl in CFG["templates"]:
        pool = sorted(q["query_id"] for q in cand if q["template"] == tpl and q["group"] == "stable")
        keep |= set(random.Random(f"{SEED}-{tpl}-stable").sample(pool, min(CFG["stable_cap"], len(pool))))
    cand = [q for q in cand if q["group"] != "stable" or q["query_id"] in keep]

BASE, EV = [], {}
for q in cand:
    it = ITEMS[q["item_id"]]
    eo, en = q["evidence"][V_OLD], evidence_new(q, it)
    assert eo and en, f"{q['query_id']}: thiếu bằng chứng O hoặc N"
    assert all(e in CORP[V_OLD] for e in eo) and all(e in CORP[V_NEW] for e in en)
    BASE.append(q)
    EV[q["query_id"]] = (eo, en)

# Kiểm tra: đoạn O chứa đáp án bản cũ (nhóm changed), đoạn N chứa đáp án bản mới
bad = []
for q in BASE:
    eo, en = EV[q["query_id"]]
    t_old = "\n".join(CORP[V_OLD][e]["text"] for e in eo)
    t_new = "\n".join(CORP[V_NEW][e]["text"] for e in en)
    if q["group"] == "changed" and not all(re.search(re.escape(g) + r"(?![.\d])", t_old) for g in q["other_gold"]):
        bad.append((q["query_id"], "O"))
    if not all(re.search(re.escape(g) + r"(?![.\d])", t_new) for g in q["gold"]):
        bad.append((q["query_id"], "N"))
assert not bad, f"Bằng chứng không chứa đáp án: {bad[:10]}"

QUERIES = []
for q in BASE:
    QUERIES.append({**q, "base_id": q["query_id"], "qform": "ver"})
    qn = nover(q["question"])
    assert qn != q["question"], f"{q['query_id']}: không bỏ được số phiên bản khỏi câu hỏi"
    assert V_NEW not in qn and not re.search(r"\bv\d", qn) and "current" in qn, f"{q['query_id']}: câu nover sai: {qn}"
    QUERIES.append({**q, "question": qn, "base_id": q["query_id"], "query_id": q["query_id"] + "_nover", "qform": "nover"})
QBYID = {q["query_id"]: q for q in QUERIES}
assert len({q["question"] for q in QUERIES}) == len(QUERIES), "có hai truy vấn trùng câu hỏi"
with open(OUT / "queries.jsonl", "w", encoding="utf-8") as f:
    for q in QUERIES:
        f.write(json.dumps(q, ensure_ascii=False, sort_keys=True) + "\n")
REVOKED = {q["query_id"]: ITEMS[q["item_id"]]["anchor_status_new"] != "active" for q in BASE}
print(f"{len(BASE)} câu gốc:", dict(collections.Counter((q['template'][:2], q['group']) for q in BASE)))
print("Câu changed có mã bị thu hồi:", dict(collections.Counter(q["template"][:2] for q in BASE
                                                             if q["group"] == "changed" and REVOKED[q["query_id"]])))
print("Số cụm dòng dõi (nhóm changed):", len({q["lineage"] for q in BASE if q["group"] == "changed"}))

361 câu gốc: {('D1', 'stable'): 80, ('D1', 'changed'): 72, ('D4', 'stable'): 80, ('D4', 'changed'): 129}
Câu changed có mã bị thu hồi: {'D1': 62, 'D4': 129}
Số cụm dòng dõi (nhóm changed): 133


## 3. Đoạn nhiễu và ghi chú cầu nối

Dùng đúng cách của notebook 05 và 06:

- **Đoạn nhiễu:** BM25 trên kho bản mới theo câu hỏi `ver`, loại mọi đoạn có mã hoặc tên tactic của đáp án hai phiên bản, kể cả khi tên nằm trong một từ khác (ví dụ "ApplePersistence"). Mỗi câu có một danh sách đoạn nhiễu cố định, dùng chung cho mọi điều kiện và cả hai cách hỏi.
- **Ghi chú cầu nối:** câu *"… was revoked and replaced by …"*, chỉ có ở câu có mã bị thu hồi.

In [38]:
TOKEN_RE = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

class BM25:                            # giống hệt notebook 04–06
    def __init__(self, docs, k1=1.2, b=0.75):
        self.ids = sorted(docs)
        n = len(self.ids)
        toks = [collections.Counter(tokenize(docs[i])) for i in self.ids]
        dl = np.array([sum(t.values()) for t in toks], dtype=float)
        avgdl = dl.mean()
        post = collections.defaultdict(lambda: ([], []))
        for j, tf in enumerate(toks):
            for t, c in tf.items():
                post[t][0].append(j)
                post[t][1].append(c)
        self.post = {}
        for t, (idx, tf) in post.items():
            idx, tf = np.array(idx), np.array(tf, dtype=float)
            idf = math.log(1 + (n - len(idx) + 0.5) / (len(idx) + 0.5))
            self.post[t] = (idx, idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl[idx] / avgdl)))
        self.n = n

    def search(self, query, k=5):
        scores = np.zeros(self.n)
        for t in sorted(set(tokenize(query))):
            if t in self.post:
                idx, w = self.post[t]
                scores[idx] += w
        cand = np.nonzero(scores)[0]
        order = cand[np.lexsort((cand, -scores[cand]))][:k]
        return [(self.ids[j], float(scores[j])) for j in order]

INDEX_NEW = BM25({i: d["title"] + "\n" + d["text"] for i, d in CORP[V_NEW].items()})
_old_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["new_only_labels"]}
_new_names = {n for n in TACTIC_ALIAS if n not in ALIAS_FILE["old_only_labels"]}
OLD_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_old_names)}
NEW_NAME = {TACTIC_ALIAS[n]: n for n in sorted(_new_names)}

def bridge_note(q):
    it = ITEMS[q["item_id"]]
    if q["group"] != "changed" or it["anchor_status_new"] == "active":
        return None
    major = V_NEW.split(".")[0]
    # Chỉ tách tên khi có nhiều mã kế nhiệm (tên như "Registry Run Keys / Startup Folder" có sẵn " / ")
    names = it["name_new"].split(" / ") if len(it["anchor_new"]) > 1 else [it["name_new"]]
    assert len(names) == len(it["anchor_new"]), f"{it['item_id']}: số tên và số mã kế nhiệm không khớp"
    repl = ", ".join(f"{s} ({n})" for s, n in zip(it["anchor_new"], names))
    return (f"In MITRE ATT&CK v{major}, technique {it['anchor_old']} ({it['name_old']}) was revoked "
            f"and replaced by {repl}.")

def banned_terms(q):
    it = ITEMS[q["item_id"]]
    ids = {it["anchor_old"], *it["anchor_new"], *q["gold"], *q["other_gold"]}
    names = set()
    if q["answer_type"] == "tactic_set":
        for t in set(q["gold"]) | set(q["other_gold"]):
            names |= {OLD_NAME.get(t), NEW_NAME.get(t)} - {None}
    return ids, names

def mentions_answer(text, ids, names):
    # Có mã đáp án (không tính khi chỉ là tiền tố của mã con) hoặc tên tactic đáp án, kể cả khi nằm trong từ khác
    # (ví dụ "ApplePersistence"). Bản 1.3: bộ lọc dùng đúng tiêu chí của phép kiểm tra rò rỉ bên dưới.
    low = text.lower()
    return any(re.search(re.escape(i) + r"(?![.\d])", text) for i in ids) or any(n.lower() in low for n in names)

def distractors(q, k):
    ids, names = banned_terms(q)
    eo, en = EV[q["query_id"]]
    out = []
    for cid, _ in INDEX_NEW.search(q["question"], 3000):
        d = CORP[V_NEW][cid]
        if cid in en or d["metadata"]["doc_id"] in ids or mentions_answer(d["text"], ids, names):
            continue
        out.append(cid)
        if len(out) == k:
            break
    assert len(out) == k, f"{q['query_id']}: không đủ đoạn nhiễu"
    return out

BRIDGE   = {q["query_id"]: bridge_note(q) for q in BASE}
DISTRACT = {q["query_id"]: distractors(q, N_PASSAGES) for q in BASE}
leaks = []
for q in BASE:
    ids, names = banned_terms(q)
    for cid in DISTRACT[q["query_id"]]:
        t = CORP[V_NEW][cid]["text"]
        if any(re.search(re.escape(i) + r"(?![.\d])", t) for i in ids) or any(n.lower() in t.lower() for n in names):
            leaks.append((q["query_id"], cid))
assert not leaks, f"Đoạn nhiễu lộ đáp án: {leaks[:5]}"
assert all(BRIDGE[k] for k, v in REVOKED.items() if v and QBYID[k]["group"] == "changed")
print(f"✓ Đoạn nhiễu không lộ đáp án; {sum(1 for v in BRIDGE.values() if v)} câu gốc có ghi chú cầu nối")

✓ Đoạn nhiễu không lộ đáp án; 191 câu gốc có ghi chú cầu nối


## 4. Prompt

Prompt giống hệt notebook 04–06: cùng lời dặn hệ thống, cùng yêu cầu theo kiểu đáp án, cùng lược đồ JSON. Có hai bộ prompt:

- `prompts/`: lời dặn ở vai trò `system`;
- `prompts_mergedsys/`: lời dặn được ghép vào đầu lượt người dùng, dùng cho Gemma-2 và Mistral.

Ô kiểm tra xác nhận các điều sau:

- bố cục, nhãn và tráo nhãn đúng;
- hai cách hỏi dùng cùng ngữ cảnh;
- bộ prompt ghép lời dặn có cùng nội dung với bộ prompt chuẩn;
- **ở nhánh A**, mọi prompt trùng **từng ký tự** với prompt của notebook 05 (`new`, `old`) và notebook 06 (các điều kiện còn lại).

In [39]:
SYSTEM = ("You are a cyber threat intelligence (CTI) assistant with expertise in the MITRE ATT&CK Enterprise framework. "
          "Answer the user's question accurately. Respond only with a JSON object of the form {\"answer\": [...]}. "
          "Put only the requested items in the list. Use an empty list if you do not know.")
INSTR = {
    "tactic_set":     'List each tactic as its ATT&CK tactic ID followed by its name, for example "TA0001 Initial Access".',
    "technique_set":  "List ATT&CK technique IDs only (format T1234 or T1234.001).",
    "mitigation_set": "List ATT&CK mitigation IDs only (format M1234).",
}
PATTERN = {"technique_set": "^T[0-9]{4}(\\.[0-9]{3})?$", "mitigation_set": "^M[0-9]{4}$",
           "tactic_set": "^[A-Za-z0-9 ,.()&/'-]{1,60}$"}
TAGS = {
    "full": {V_OLD: f"[Source: MITRE ATT&CK Enterprise v{V_OLD} (superseded)]",
             V_NEW: f"[Source: MITRE ATT&CK Enterprise v{V_NEW} (current release)]"},
    "ver":  {V_OLD: f"[Source: MITRE ATT&CK Enterprise v{V_OLD}]",
             V_NEW: f"[Source: MITRE ATT&CK Enterprise v{V_NEW}]"},
}
SWAP_VER = {V_OLD: V_NEW, V_NEW: V_OLD}

def schema_for(answer_type):
    return {"type": "object",
            "properties": {"answer": {"type": "array", "items": {"type": "string", "pattern": PATTERN[answer_type]},
                                      "maxItems": 8}},
            "required": ["answer"], "additionalProperties": False}

def clip(text, n):
    words = list(re.finditer(r"\S+", text))
    return text if len(words) <= n else text[:words[n - 1].end()] + " …"

def passages(q, cond):
    spec, bid = CONDITIONS[cond], q["base_id"]
    eo, en = EV[bid]
    out = []
    for kind in spec["order"]:
        if kind == "O":
            out += [("O", V_OLD, CORP[V_OLD][e]["text"]) for e in eo]
        elif kind == "N":
            out += [("N", V_NEW, CORP[V_NEW][e]["text"]) for e in en]
        elif kind == "B" and BRIDGE[bid]:
            out.append(("B", None, BRIDGE[bid]))
    out += [("D", V_NEW, CORP[V_NEW][d]["text"]) for d in DISTRACT[bid][:max(0, N_PASSAGES - len(out))]]
    res = []
    for k, v, t in out:
        label = None
        if spec["tag"] and v:
            label = TAGS[spec["tag"]][SWAP_VER[v] if (spec["swap"] and k in ("O", "N")) else v]
        res.append((k, v, label, (label + "\n" if label else "") + clip(t, MAX_CHUNK_WORDS)))
    return res

def build_messages(q, cond):
    ctx = "\n\n".join(f"[{i + 1}] {t}" for i, (_, _, _, t) in enumerate(passages(q, cond)))
    user = ("Use the following context passages from a CTI knowledge base if they are relevant.\n\n"
            f"{ctx}\n\nQuestion: {q['question']}\n{INSTR[q['answer_type']]}")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def merge_system(msgs):
    return [{"role": "user", "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]

def cond_queries(cond):
    return [q for q in QUERIES if q["qform"] in CONDITIONS[cond]["qforms"]]

for cond in CONDITIONS:
    with open(PROMPTS / f"{cond}.jsonl", "w", encoding="utf-8") as f1, \
         open(PROMPTS_MS / f"{cond}.jsonl", "w", encoding="utf-8") as f2:
        for q in cond_queries(cond):
            msgs = build_messages(q, cond)
            row = dict(query_id=q["query_id"], condition=cond, answer_type=q["answer_type"], messages=msgs,
                       schema=schema_for(q["answer_type"]), layout=[k for k, *_ in passages(q, cond)])
            f1.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
            f2.write(json.dumps({**row, "messages": merge_system(msgs)}, ensure_ascii=False, sort_keys=True) + "\n")
MANIFEST["prompts"] = {c: sha256(PROMPTS / f"{c}.jsonl") for c in CONDITIONS}
MANIFEST["prompts_mergedsys"] = {c: sha256(PROMPTS_MS / f"{c}.jsonl") for c in CONDITIONS}
print(pd.Series({c: (len(cond_queries(c)), collections.Counter(tuple(k for k, *_ in passages(q, c))
                                                               for q in cond_queries(c)).most_common(2))
                 for c in CONDITIONS}).to_string())

new                            (361, [(('N', 'D', 'D', 'D'), 361)])
old                            (361, [(('O', 'D', 'D', 'D'), 361)])
old_new                        (361, [(('O', 'N', 'D', 'D'), 361)])
new_old                        (361, [(('N', 'O', 'D', 'D'), 361)])
tag_old_new                    (361, [(('O', 'N', 'D', 'D'), 361)])
tag_new_old                    (361, [(('N', 'O', 'D', 'D'), 361)])
vtag_old_new                   (722, [(('O', 'N', 'D', 'D'), 722)])
vtag_new_old                   (722, [(('N', 'O', 'D', 'D'), 722)])
swap_old_new                   (361, [(('O', 'N', 'D', 'D'), 361)])
swap_new_old                   (361, [(('N', 'O', 'D', 'D'), 361)])
bridge_old_new    (361, [(('O', 'N', 'B', 'D'), 191), (('O', 'N'...
bridge_new_old    (361, [(('N', 'O', 'B', 'D'), 191), (('N', 'O'...


In [40]:
errs = []
for q in QUERIES:
    bid = q["base_id"]
    eo, en = EV[bid]
    for cond, spec in CONDITIONS.items():
        ps = passages(q, cond)
        exp = []
        for k in spec["order"]:
            exp += ["O"] * len(eo) if k == "O" else ["N"] * len(en) if k == "N" else (["B"] if BRIDGE[bid] else [])
        kinds = [k for k, *_ in ps]
        if kinds[:len(exp)] != exp or set(kinds[len(exp):]) - {"D"} or len(ps) != max(N_PASSAGES, len(exp)):
            errs.append((q["query_id"], cond, "bố cục"))
        text = build_messages(q, cond)[1]["content"]
        for style, tags in TAGS.items():
            if any(t in text for t in tags.values()) != (spec["tag"] == style):
                errs.append((q["query_id"], cond, f"nhãn kiểu {style}"))
        if spec["swap"]:
            for k, v, label, _ in ps:
                if k == "O" and label != f"[Source: MITRE ATT&CK Enterprise v{V_NEW} (current release)]":
                    errs.append((q["query_id"], cond, "tráo nhãn O"))
                if k == "N" and label != f"[Source: MITRE ATT&CK Enterprise v{V_OLD} (superseded)]":
                    errs.append((q["query_id"], cond, "tráo nhãn N"))
    if q["qform"] == "nover":
        a = build_messages(q, "vtag_old_new")[1]["content"].split("\n\nQuestion:")[0]
        b = build_messages(QBYID[bid], "vtag_old_new")[1]["content"].split("\n\nQuestion:")[0]
        if a != b:
            errs.append((q["query_id"], "*", "ngữ cảnh ver/nover khác nhau"))
assert not errs, errs[:10]
for cond in CONDITIONS:                                   # bộ prompt ghép lời dặn: cùng nội dung
    for r1, r2 in zip(map(json.loads, open(PROMPTS / f"{cond}.jsonl", encoding="utf-8")),
                      map(json.loads, open(PROMPTS_MS / f"{cond}.jsonl", encoding="utf-8"))):
        assert r1["query_id"] == r2["query_id"] and len(r2["messages"]) == 1 and r2["messages"][0]["role"] == "user"
        assert r2["messages"][0]["content"] == r1["messages"][0]["content"] + "\n\n" + r1["messages"][1]["content"]
print(f"✓ {sum(len(cond_queries(c)) for c in CONDITIONS)} prompt đúng cấu trúc (mỗi bộ)")

# Nhánh A: trùng prompt notebook 05 (new, old) và notebook 06 (còn lại)
if CFG["branch"] == "A":
    srcs = {"new": ROOT / "results/ctx-18.1-19.2/prompts", "old": ROOT / "results/ctx-18.1-19.2/prompts"}
    for c in CONDITIONS:
        srcs.setdefault(c, ROOT / "results/mech-18.1-19.2/prompts")
    n_cmp, skipped = 0, []
    for c, d in srcs.items():
        if not (d / f"{c}.jsonl").exists():
            skipped.append(c); continue
        old = {r["query_id"]: r["messages"] for r in map(json.loads, open(d / f"{c}.jsonl", encoding="utf-8"))}
        new = {r["query_id"]: r["messages"] for r in map(json.loads, open(PROMPTS / f"{c}.jsonl", encoding="utf-8"))}
        if c in ("new", "old"):        # notebook 05 có cả cách hỏi unversioned: chỉ so phần current
            old = {k: v for k, v in old.items() if k.endswith("|current")}
        else:                          # notebook 06 có cả hai cách hỏi cho mọi điều kiện: chỉ so cách hỏi dùng ở đây
            qf = lambda k: "nover" if k.endswith("_nover") else "ver"
            old = {k: v for k, v in old.items() if qf(k) in CONDITIONS[c]["qforms"]}
        assert set(old) == set(new), f"{c}: tập câu hỏi khác notebook 05/06 ({len(set(old) ^ set(new))} câu lệch)"
        for k in new:
            assert old[k] == new[k], f"Prompt {c}/{k} khác notebook 05/06 — gửi thầy"
            n_cmp += 1
    assert not skipped, f"Thiếu prompt notebook 05/06 để đối chiếu: {skipped}"
    print(f"✓ {n_cmp} prompt trùng từng ký tự với notebook 05/06")

✓ 5054 prompt đúng cấu trúc (mỗi bộ)


In [41]:
smoke_ids = set()
for key in sorted({(q["template"], q["group"], q["qform"]) for q in QUERIES}):
    same = [q["query_id"] for q in QUERIES if (q["template"], q["group"], q["qform"]) == key]
    smoke_ids |= set(same[:SMOKE_PER_GROUP])
for src, dst in ((PROMPTS, SMOKE_P), (PROMPTS_MS, SMOKE_P_MS)):
    for cond in CONDITIONS:
        with open(src / f"{cond}.jsonl", encoding="utf-8") as fi, open(dst / f"{cond}.jsonl", "w", encoding="utf-8") as fo:
            for line in fi:
                if json.loads(line)["query_id"] in smoke_ids:
                    fo.write(line)
print(f"Bộ chạy thử: {len(smoke_ids)} truy vấn")

Bộ chạy thử: 8 truy vấn


## 5. Chạy suy luận

Như notebook 05 và 06: mỗi mô hình chạy trong một tiến trình riêng.

- **Chạy thử** trước cho mọi mô hình, để phát hiện sớm lỗi tải mô hình hay lỗi khuôn hội thoại.
- **Thời gian ước tính** trên RTX 4070: khoảng 15–25 phút mỗi mô hình. Nhánh A có 3 mô hình, nhánh B có 5. Lần đầu còn thêm thời gian tải mô hình mới.
- **Theo dõi tiến độ:** lệnh `tail -f …` được in ra để em xem trong terminal. Ô sẽ im lặng trong lúc chạy.
- **Mô hình đã chạy xong** (cùng prompt, cùng cấu hình) sẽ được bỏ qua nếu chạy lại ô.
- **Nếu một mô hình chạy lỗi**, notebook tự thử các cấu hình môi trường dự phòng (xem "Mô hình lỗi kỹ thuật" ở phần kế hoạch). Mỗi lần thử hỏng mất khoảng 5 phút. Log chạy ở cấu hình k ≥ 1 có đuôi `_r<k>`, ví dụ `gemma2-9b_smoke_r1.log`.

Ô đầu tiên ghi module suy luận bản 2 và kiểm tra mã băm của nó.

In [42]:
INFER_SRC = r'''# Chạy suy luận cho MỘT mô hình (bản 2, notebook 07). Gọi từ notebook qua subprocess.
# Khác bản 1 (notebook 04): thêm --engine-kwargs (tham số máy vLLM cho cấu hình môi trường dự phòng),
# ghi biến môi trường BLAS vào khoá chạy và meta, in bộ nhớ GPU trước khi nạp mô hình.
import argparse, hashlib, json, os, sys, time, platform, subprocess
from pathlib import Path

os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")


def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]


def done_ok(path, meta_path, ids, run_key):
    # Kết quả cũ chỉ được dùng lại khi: đủ đúng các query_id theo đúng thứ tự VÀ cùng "khoá chạy"
    # (cùng file prompt, cùng mô hình/phiên bản, cùng tham số sinh). Đổi bất cứ thứ gì -> chạy lại.
    if not path.exists() or not meta_path.exists():
        return False
    try:
        got = [r["query_id"] for r in read_jsonl(path)]
        key = json.loads(meta_path.read_text()).get("run_key")
    except (ValueError, KeyError):
        return False
    return got == ids and key == run_key


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--revision", required=True)
    ap.add_argument("--tag", required=True)
    ap.add_argument("--prompts-dir", required=True)
    ap.add_argument("--conditions", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--max-model-len", type=int, default=6144)
    ap.add_argument("--max-tokens", type=int, default=96)
    ap.add_argument("--gpu-mem", type=float, default=0.85)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--chat-template-kwargs", default="{}")
    ap.add_argument("--engine-kwargs", default="{}")
    a = ap.parse_args()

    conds = [c for c in a.conditions.split(",") if c]
    out_dir = Path(a.out_dir) / a.tag
    out_dir.mkdir(parents=True, exist_ok=True)
    ctk = json.loads(a.chat_template_kwargs)
    ek = json.loads(a.engine_kwargs)
    assert isinstance(ek, dict) and set(ek) <= {"enforce_eager", "max_num_seqs", "max_num_batched_tokens"}, ek
    blas_env = os.environ.get("TORCH_BLAS_PREFER_CUBLASLT", "")

    from importlib.metadata import version as pkg_version
    script_sha = sha256_file(Path(__file__))
    vllm_ver = pkg_version("vllm")
    jobs, keys = {}, {}
    for c in conds:
        pfile = Path(a.prompts_dir) / f"{c}.jsonl"
        rows = read_jsonl(pfile)
        if a.limit:
            rows = rows[:a.limit]
        ids = [r["query_id"] for r in rows]
        keys[c] = hashlib.sha256(json.dumps(dict(prompts=sha256_file(pfile), limit=a.limit, repo=a.repo,
                                                 revision=a.revision, max_model_len=a.max_model_len,
                                                 max_tokens=a.max_tokens, seed=a.seed, ctk=ctk, gpu_mem=a.gpu_mem,
                                                 script=script_sha, vllm=vllm_ver, engine=ek, blas_env=blas_env),
                                            sort_keys=True).encode()).hexdigest()
        if done_ok(out_dir / f"{c}.jsonl", out_dir / f"meta_{c}.json", ids, keys[c]):
            print(f"[{a.tag}] {c}: đã có đủ {len(ids)} kết quả cùng cấu hình — bỏ qua", flush=True)
            continue
        jobs[c] = rows
    if not jobs:
        print(f"[{a.tag}] không còn việc gì để chạy", flush=True)
        return

    from huggingface_hub import snapshot_download
    path = snapshot_download(a.repo, revision=a.revision)
    print(f"[{a.tag}] mô hình tại {path}", flush=True)

    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    budget = a.max_model_len - a.max_tokens
    for c, rows in jobs.items():
        lens = []
        for r in rows:
            text = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True, **ctk)
            lens.append(len(tok(text, add_special_tokens=False)["input_ids"]))
        too_long = [(r["query_id"], n) for r, n in zip(rows, lens) if n > budget]
        print(f"[{a.tag}] {c}: {len(rows)} prompt, dài nhất {max(lens)} token (giới hạn {budget})", flush=True)
        if too_long:
            sys.exit(f"LỖI: {len(too_long)} prompt vượt {budget} token, ví dụ {too_long[:3]} — tăng MAX_MODEL_LEN "
                     "hoặc giảm MAX_CHUNK_WORDS trong notebook")

    try:   # nvidia-smi không khởi tạo CUDA trong tiến trình này (vLLM cần điều đó)
        smi = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception as e:
        smi = f"không đọc được ({type(e).__name__})"
    print(f"[{a.tag}] GPU trước khi nạp: {smi}; TORCH_BLAS_PREFER_CUBLASLT={blas_env or '-'}; engine={ek}", flush=True)

    import torch
    import vllm
    from vllm import LLM, SamplingParams
    from vllm.sampling_params import StructuredOutputsParams

    try:   # đọc ở tiến trình này; EngineCore thừa hưởng cùng biến môi trường
        blas_backend = str(torch.backends.cuda.preferred_blas_library())
    except Exception as e:
        blas_backend = f"? ({type(e).__name__})"
    # disable_any_whitespace: JSON dạng gọn, mô hình không thể sinh khoảng trắng vô hạn rồi bị cắt ở max_tokens
    llm = LLM(model=path, tokenizer=path, max_model_len=a.max_model_len,
              gpu_memory_utilization=a.gpu_mem, seed=a.seed,
              structured_outputs_config={"backend": "xgrammar", "disable_any_whitespace": True}, **ek)
    meta_base = dict(model_tag=a.tag, repo=a.repo, revision=a.revision, snapshot=Path(path).name,
                     vllm=vllm.__version__, torch=torch.__version__, cuda=torch.version.cuda,
                     gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
                     max_model_len=a.max_model_len, max_tokens=a.max_tokens, gpu_mem=a.gpu_mem, seed=a.seed,
                     chat_template_kwargs=ctk, flashinfer_sampler=os.environ.get("VLLM_USE_FLASHINFER_SAMPLER"),
                     script_sha256=script_sha, structured_outputs="xgrammar, disable_any_whitespace",
                     engine_kwargs=ek, blas_env=blas_env, blas_backend_parent=blas_backend)

    for c, rows in jobs.items():
        sps = [SamplingParams(temperature=0.0, max_tokens=a.max_tokens, seed=a.seed,
                              structured_outputs=StructuredOutputsParams(json=r["schema"])) for r in rows]
        t0 = time.time()
        outs = llm.chat([r["messages"] for r in rows], sps, use_tqdm=True, chat_template_kwargs=ctk or None)
        dt = time.time() - t0
        assert len(outs) == len(rows)
        tmp = out_dir / f".{c}.jsonl.tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            for r, o in zip(rows, outs):
                f.write(json.dumps(dict(query_id=r["query_id"], condition=c, model_tag=a.tag,
                                        text=o.outputs[0].text, finish_reason=o.outputs[0].finish_reason,
                                        n_prompt_tokens=len(o.prompt_token_ids or []),
                                        n_output_tokens=len(o.outputs[0].token_ids)), ensure_ascii=False) + "\n")
        os.replace(tmp, out_dir / f"{c}.jsonl")
        (out_dir / f"meta_{c}.json").write_text(json.dumps({**meta_base, "condition": c, "n": len(rows),
                                                            "run_key": keys[c], "seconds": round(dt, 1)}, indent=2))
        print(f"[{a.tag}] {c}: xong {len(rows)} truy vấn trong {dt / 60:.1f} phút", flush=True)


if __name__ == "__main__":
    main()
'''

INFER = ROOT / "ctidrift_infer_v2.py"
INFER.write_text(INFER_SRC, encoding="utf-8")
assert sha256(INFER) == "39b80fe37bb06d4ddbe082a6c372258afe10967a0a599539cd2449d822ea4f74", "ctidrift_infer_v2.py bị sửa — gửi thầy"
MANIFEST["inputs"]["infer_module"] = {"path": INFER.name, "sha256": sha256(INFER)}
print(f"✓ Đã ghi {INFER.name} ({sha256(INFER)[:12]}…)")

✓ Đã ghi ctidrift_infer_v2.py (39b80fe37bb0…)


In [43]:
class InferError(RuntimeError):
    pass

def effective(model, k):
    # Giá trị thực dùng ở cấu hình k: gpu_mem và max_model_len (không vượt giới hạn riêng của mô hình)
    R = ENV_LADDER[k]
    mml = model.get("max_model_len", MAX_MODEL_LEN)
    return dict(gpu_mem=R["gpu_mem"] or GPU_MEM, max_model_len=min(mml, R["max_model_len"]) if R["max_model_len"] else mml)

def _group_alive(pgid):
    # Còn tiến trình nào (không phải zombie) trong nhóm pgid không? Đọc /proc (Linux)
    for d in os.listdir("/proc"):
        if not d.isdigit():
            continue
        try:
            with open(f"/proc/{d}/stat") as fh:
                st = fh.read()
        except OSError:
            continue
        rest = st[st.rindex(")") + 2:].split()      # trạng thái, ppid, pgrp, …
        if int(rest[2]) == pgid and rest[0] not in ("Z", "X"):
            return True
    return False

def run_infer(model, out_dir, smoke=False, rung=0):
    R = ENV_LADDER[rung]
    pdir = (SMOKE_P_MS if smoke else PROMPTS_MS) if model["merge_system"] else (SMOKE_P if smoke else PROMPTS)
    log = LOGS / f"{model['tag']}{'_smoke' if smoke else ''}{f'_r{rung}' if rung else ''}.log"
    eff = effective(model, rung)
    cmd = [sys.executable, INFER.name, "--repo", model["repo"], "--revision", model["revision"],
           "--tag", model["tag"], "--prompts-dir", str(pdir), "--conditions", ",".join(CONDITIONS),
           "--out-dir", str(out_dir), "--max-model-len", str(eff["max_model_len"]),
           "--max-tokens", str(MAX_TOKENS),
           "--gpu-mem", str(eff["gpu_mem"]), "--seed", str(SEED),
           "--chat-template-kwargs", json.dumps(model["chat_template_kwargs"]),
           "--engine-kwargs", json.dumps(R["engine"], sort_keys=True)]
    env = {**ENV, **R["env"]}
    print(f"▶ {model['tag']} [cấu hình {rung}: {R['name']}]: đang chạy… "
          f"theo dõi tiến độ trong terminal bằng lệnh:\n    tail -f {log}", flush=True)
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, cwd=ROOT, start_new_session=True)
        def _kill(sig):
            try:
                os.killpg(p.pid, sig)
            except ProcessLookupError:
                pass
        try:
            p.wait()
        except BaseException:
            _kill(signal.SIGTERM)
            try:
                p.wait(timeout=60)
            except BaseException:
                pass
            _kill(signal.SIGKILL)
            print("⏹ Đã dừng và giải phóng GPU. Kiểm tra lại bằng lệnh nvidia-smi nếu cần.")
            raise
    # Dọn tiến trình con còn sót (EngineCore) và chờ chúng thoát hẳn, để lần chạy sau có đủ bộ nhớ GPU
    _kill(signal.SIGKILL)
    t1 = time.time()
    while _group_alive(p.pid) and time.time() - t1 < 120:
        time.sleep(1)
    if _group_alive(p.pid):
        print(f"⚠ Tiến trình của {model['tag']} chưa thoát hẳn sau 120 giây — kiểm tra bằng nvidia-smi")
    time.sleep(3)
    text = log.read_text(encoding="utf-8", errors="replace")
    tail = text.splitlines()
    print("\n".join(l for l in tail if l.startswith(f"[{model['tag']}]") or "LỖI" in l or "Error" in l)[-3000:])
    print(f"◀ {model['tag']}: {'XONG' if p.returncode == 0 else 'LỖI'} sau {(time.time() - t0) / 60:.1f} phút")
    if p.returncode != 0:
        print("\n".join(tail[-40:]))
        if "prompt vượt" in text:      # lỗi cấu hình của notebook, không phải lỗi môi trường: dừng hẳn
            raise AssertionError(f"Prompt dài quá giới hạn — gửi thầy file {log}")
        raise InferError(f"Suy luận lỗi — gửi thầy file {log}")

MBYTAG = {m["tag"]: m for m in ALL_MODELS}

def try_ladder(model, out_dir, start, smoke):
    # Thử từ cấu hình `start` trở lên; trả về chỉ số cấu hình đầu tiên chạy được, hoặc None
    for k in range(start, len(ENV_LADDER)):
        rec = dict(phase="chạy thử" if smoke else "chạy chính", rung=k)
        try:
            run_infer(model, out_dir, smoke=smoke, rung=k)
        except InferError as e:
            ATTEMPTS.setdefault(model["tag"], []).append({**rec, "ok": False, "error": str(e)})
            if model["tag"] in RUNG and k + 1 < len(ENV_LADDER):
                RUNG[model["tag"]] = k + 1          # chạy chính đã lên thang: ghi ngay, để lần chạy lại không quay xuống
            save_env_rung()
            if k + 1 < len(ENV_LADDER):
                print(f"↻ {model['tag']}: thử cấu hình dự phòng {k + 1} ({ENV_LADDER[k + 1]['name']})", flush=True)
            continue
        ATTEMPTS.setdefault(model["tag"], []).append({**rec, "ok": True})
        save_env_rung()
        return k
    return None

def save_env_rung():
    cur = {**START, **RUNG}      # mô hình chưa chạy tới trong lần này giữ cấu hình bắt đầu (không quay xuống nếu bị ngắt)
    MANIFEST["env_rung"] = {t: dict(rung=k, **ENV_LADDER[k], effective=effective(MBYTAG[t], k)) for t, k in cur.items()}
    MANIFEST["env_attempts"] = ATTEMPTS
    tmp = OUT / ".env_rung.json.tmp"           # ghi file tạm rồi đổi tên: không bao giờ để lại file dở
    tmp.write_text(json.dumps(dict(rung=MANIFEST["env_rung"], start=START, attempts=ATTEMPTS), indent=2, ensure_ascii=False))
    os.replace(tmp, OUT / "env_rung.json")

def read_rungs(p):
    # env_rung.json -> ({mô hình: cấu hình}, các lần thử); file chưa có -> rỗng
    if not p.exists():
        return {}, {}
    d = json.loads(p.read_text())
    return {t: v["rung"] for t, v in d.get("rung", {}).items()}, d.get("attempts", {})

# Cấu hình bắt đầu của mỗi mô hình (chỉ đi lên thang, không quay lại):
#  - cấu hình đã dùng ở lần chạy trước của chính nhánh này (nếu chạy lại);
#  - ở nhánh B: không thấp hơn cấu hình mô hình đó đã dùng ở nhánh A, để C7 so sánh trên cùng cấu hình.
START, ATTEMPTS = read_rungs(OUT / "env_rung.json")
if START:
    print("Lần chạy trước của nhánh này đã dùng cấu hình:", START)
if CFG["branch"] == "B":
    _rA = ROOT / "results" / "conf-18.1-19.2" / "env_rung.json"
    if _rA.exists():
        for t, k in read_rungs(_rA)[0].items():
            START[t] = max(START.get(t, 0), k)
        print("Cấu hình bắt đầu sau khi xét nhánh A:", START)
    else:
        print("⚠ Chưa có env_rung.json của nhánh A — theo kế hoạch, nên chạy nhánh A trước nhánh B")

JSON_MIN = 0.95                    # dưới mức này: mô hình lỗi kỹ thuật (định trước)
FAILED = {}                        # mô hình lỗi kỹ thuật -> lý do
RUNG = {}                          # mô hình -> chỉ số cấu hình môi trường đang dùng
# ATTEMPTS: mô hình -> các lần thử (giai đoạn, cấu hình, kết quả), nối tiếp các lần chạy trước
SMOKE = OUT / "raw_smoke"
for m in MODELS:
    k0 = START.get(m["tag"], 0)
    k = try_ladder(m, SMOKE, k0, smoke=True)
    if k is None:
        FAILED[m["tag"]] = f"chạy thử lỗi ở mọi cấu hình từ {k0} tới {len(ENV_LADDER) - 1}"
        continue
    RUNG[m["tag"]] = k
    fmt = [score_drift(QBYID[r["query_id"]], r["text"], TACTIC_ALIAS)["format"] == "json"
           for cond in CONDITIONS for r in map(json.loads, open(SMOKE / m["tag"] / f"{cond}.jsonl", encoding="utf-8"))]
    if np.mean(fmt) < JSON_MIN:
        FAILED[m["tag"]] = f"chạy thử: tỉ lệ JSON {np.mean(fmt):.2f} < {JSON_MIN}"
save_env_rung()
for t, k in RUNG.items():
    print(f"  {t:16s} cấu hình {k}: {ENV_LADDER[k]['name']}")
if FAILED:
    print("⚠⚠ CÓ MÔ HÌNH LỖI KỸ THUẬT — gửi thầy file log trước khi chạy tiếp:", FAILED)
else:
    print("✓ Chạy thử: mọi mô hình chạy được và trả lời đúng định dạng JSON")

Cấu hình bắt đầu sau khi xét nhánh A: {'gemma2-9b': 0, 'phi4-mini': 0, 'mistral7b-v0.3': 0}
▶ qwen3-8b [cấu hình 0: mặc định]: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/conf-6.3-7.2/logs/qwen3-8b_smoke.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] new: 4 prompt, dài nhất 1356 token (giới hạn 5984)
[qwen3-8b] old: 4 prompt, dài nhất 1361 token (giới hạn 5984)
[qwen3-8b] old_new: 4 prompt, dài nhất 1108 token (giới hạn 5984)
[qwen3-8b] new_old: 4 prompt, dài nhất 1108 token (giới hạn 5984)
[qwen3-8b] tag_old_new: 4 prompt, dài nhất 1177 token (giới hạn 5984)
[qwen3-8b] tag_new_old: 4 prompt, dài nhất 1177 token (giới hạn 5984)
[qwen3-8b] vtag_old_new: 8 prompt, dài nhất 1164 token (giới hạn 5984)
[qwen3-8b] vtag_new_old: 8 prompt, dài nhất 1164 token (giới hạn 5984)
[qwen3-8b] swap_old_new: 4 prompt, dài nhất 1177 token (giới hạn 5984)
[qwen3-8b] swap_new_old: 

In [44]:
for m in MODELS:
    t = m["tag"]
    if t in FAILED:
        print(f"⏭ Bỏ qua {t} (lỗi kỹ thuật: {FAILED[t]})")
        continue
    assert t in RUNG, f"Chưa có kết quả chạy thử của {t} — chạy lại ô chạy thử ở trên trước"
    k = try_ladder(m, RAW, RUNG[t], smoke=False)
    if k is None:
        FAILED[t] = f"chạy chính lỗi ở mọi cấu hình từ {RUNG[t]} tới {len(ENV_LADDER) - 1}"
    else:
        RUNG[t] = k
save_env_rung()
(OUT / "tech_fail.json").write_text(json.dumps(FAILED, indent=2, ensure_ascii=False))
print("Cấu hình dùng cho chạy chính:", {t: f"{k}: {ENV_LADDER[k]['name']}" for t, k in RUNG.items() if t not in FAILED})
print("Mô hình lỗi kỹ thuật:", FAILED or "không có")

▶ qwen3-8b [cấu hình 0: mặc định]: đang chạy… theo dõi tiến độ trong terminal bằng lệnh:
    tail -f ~/p/cti-drift/results/conf-6.3-7.2/logs/qwen3-8b.log
[qwen3-8b] mô hình tại /data/hf/hub/models--Qwen--Qwen3-8B-AWQ/snapshots/4da05a8edb55c6046cce958586c33b61da07bb79
[qwen3-8b] new: 361 prompt, dài nhất 1700 token (giới hạn 5984)
[qwen3-8b] old: 361 prompt, dài nhất 1810 token (giới hạn 5984)
[qwen3-8b] old_new: 361 prompt, dài nhất 1801 token (giới hạn 5984)
[qwen3-8b] new_old: 361 prompt, dài nhất 1801 token (giới hạn 5984)
[qwen3-8b] tag_old_new: 361 prompt, dài nhất 1870 token (giới hạn 5984)
[qwen3-8b] tag_new_old: 361 prompt, dài nhất 1870 token (giới hạn 5984)
[qwen3-8b] vtag_old_new: 722 prompt, dài nhất 1857 token (giới hạn 5984)
[qwen3-8b] vtag_new_old: 722 prompt, dài nhất 1857 token (giới hạn 5984)
[qwen3-8b] swap_old_new: 361 prompt, dài nhất 1870 token (giới hạn 5984)
[qwen3-8b] swap_new_old: 361 prompt, dài nhất 1870 token (giới hạn 5984)
[qwen3-8b] bridge_old_new: 361 p

## 6. Chấm điểm

Mỗi câu trả lời được chấm bằng bộ chấm v1.3:

- `drift_ext` và `label_drift`: chỉ số chính, giống notebook 05 và 06;
- `stale13`: chỉ số thay thế, dùng cho phân tích độ nhạy.

In [45]:
rows = []
for m in MODELS:
    if m["tag"] in FAILED:
        continue
    for cond in CONDITIONS:
        p = RAW / m["tag"] / f"{cond}.jsonl"
        got = [json.loads(l) for l in open(p, encoding="utf-8")]
        assert [r["query_id"] for r in got] == [q["query_id"] for q in cond_queries(cond)], f"{p}: thiếu hoặc sai thứ tự"
        for r in got:
            q = QBYID[r["query_id"]]
            s = score_v13(q, r["text"], TACTIC_ALIAS)
            rows.append(dict(model=m["tag"], condition=cond, query_id=q["query_id"], base_id=q["base_id"],
                             qform=q["qform"], item_id=q["item_id"], lineage=q["lineage"],
                             revoked=REVOKED[q["base_id"]], template=q["template"], group=q["group"],
                             answer=r["text"], finish_reason=r["finish_reason"], pred="|".join(s["pred"]),
                             gold="|".join(q["gold"]), other_gold="|".join(q["other_gold"]), em=s["em"],
                             outcome=s["outcome"], drift_ext=s["drift_ext"], label_drift=s["label_drift"],
                             inconsistent=s["inconsistent"], outcome13=s["outcome13"], stale13=s["stale13"],
                             format=s["format"]))
assert rows, "Mọi mô hình đều lỗi kỹ thuật — gửi thầy các file log"
SC = pd.DataFrame(rows)
SC.to_csv(OUT / "scored.csv", index=False)
chk = SC.groupby(["model", "condition"]).agg(n=("query_id", "count"), json=("format", lambda x: (x == "json").mean()),
                                             truncated=("finish_reason", lambda x: (x == "length").sum()))
print("Tỉ lệ JSON thấp nhất:", round(chk.json.min(), 3), "| số câu bị cắt:", int(chk.truncated.sum()))
chk

Tỉ lệ JSON thấp nhất: 1.0 | số câu bị cắt: 0


n  json  truncated
model          condition                           
gemma2-9b      bridge_new_old  361   1.0          0
               bridge_old_new  361   1.0          0
               new             361   1.0          0
               new_old         361   1.0          0
               old             361   1.0          0
               old_new         361   1.0          0
               swap_new_old    361   1.0          0
               swap_old_new    361   1.0          0
               tag_new_old     361   1.0          0
               tag_old_new     361   1.0          0
               vtag_new_old    722   1.0          0
               vtag_old_new    722   1.0          0
llama3.1-8b    bridge_new_old  361   1.0          0
               bridge_old_new  361   1.0          0
               new             361   1.0          0
               new_old         361   1.0          0
               old             361   1.0          0
               old_new         361   1.0          0
               swap_new_old    361   1.0          0
               swap_old_new    361   1.0          0
               tag_new_old     361   1.0          0
               tag_old_new     361   1.0          0
               vtag_new_old    722   1.0          0
               vtag_old_new    722   1.0          0
mistral7b-v0.3 bridge_new_old  361   1.0          0
               bridge_old_new  361   1.0          0
               new             361   1.0          0
               new_old         361   1.0          0
               old             361   1.0          0
               old_new         361   1.0          0
               swap_new_old    361   1.0          0
               swap_old_new    361   1.0          0
               tag_new_old     361   1.0          0
               tag_old_new     361   1.0          0
               vtag_new_old    722   1.0          0
               vtag_old_new    722   1.0          0
phi4-mini      bridge_new_old  361   1.0          0
               bridge_old_new  361   1.0          0
               new             361   1.0          0
               new_old         361   1.0          0
               old             361   1.0          0
               old_new         361   1.0          0
               swap_new_old    361   1.0          0
               swap_old_new    361   1.0          0
               tag_new_old     361   1.0          0
               tag_old_new     361   1.0          0
               vtag_new_old    722   1.0          0
               vtag_old_new    722   1.0          0
qwen3-8b       bridge_new_old  361   1.0          0
               bridge_old_new  361   1.0          0
               new             361   1.0          0
               new_old         361   1.0          0
               old             361   1.0          0
               old_new         361   1.0          0
               swap_new_old    361   1.0          0
               swap_old_new    361   1.0          0
               tag_new_old     361   1.0          0
               tag_old_new     361   1.0          0
               vtag_new_old    722   1.0          0
               vtag_old_new    722   1.0          0

## 7. Bảng mô tả theo điều kiện

Mỗi ô là tỉ lệ kèm khoảng tin cậy 95%, bootstrap theo cụm dòng dõi. Ba nhóm câu được báo cáo:

- `changed`: lỗi thời và độ chính xác;
- `label`: tỉ lệ dùng tên cũ (chỉ có ở nhánh A);
- `stable`: độ chính xác, làm đối chứng.

In [46]:
CLUSTER = "lineage"

def _agg(df, col):
    g = df.groupby(CLUSTER)[col].agg(["sum", "count"])
    return g["sum"].to_numpy(float), g["count"].to_numpy(float), g.index

def boot_rate(df, col, B=2000):
    if len(df) == 0:
        return (float("nan"),) * 3
    s, c, _ = _agg(df, col)
    idx = np.random.default_rng(SEED).integers(0, len(s), size=(B, len(s)))
    r = s[idx].sum(1) / c[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (s.sum() / c.sum(), lo, hi)

def boot_paired(df_a, df_b, col, B=BOOT_B):
    # Hiệu ghép cặp theo cụm; trả về (hiệu, KTC 2,5%, KTC 97,5%, p một phía cho giả thuyết hiệu > 0)
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib), "hai nhóm phải có cùng tập cụm"
    assert (ca == cb).all(), "hai nhóm phải có cùng số câu trong mỗi cụm"
    if len(sa) == 0:
        return (float("nan"),) * 4
    idx = np.random.default_rng(SEED).integers(0, len(sa), size=(B, len(sa)))
    r = sa[idx].sum(1) / ca[idx].sum(1) - sb[idx].sum(1) / cb[idx].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi, (1 + (r <= 0).sum()) / (B + 1))

def signflip_p(df_a, df_b, col, B=BOOT_B):
    # p một phía (hiệu > 0) bằng phép đổi dấu ngẫu nhiên hiệu của từng cụm (ghép cặp)
    sa, ca, ia = _agg(df_a, col)
    sb, cb, ib = _agg(df_b, col)
    assert list(ia) == list(ib) and (ca == cb).all()
    if len(sa) == 0:
        return float("nan")
    d, n = sa - sb, ca.sum()
    t_obs = d.sum() / n
    signs = np.random.default_rng(SEED + 1).choice([-1.0, 1.0], size=(B, len(d)))
    t = (signs * d).sum(1) / n
    return (1 + (t >= t_obs - 1e-12).sum()) / (B + 1)

def boot_indep(df_a, df_b, col, B=BOOT_B):
    sa, ca, _ = _agg(df_a, col)
    sb, cb, _ = _agg(df_b, col)
    if len(sa) == 0 or len(sb) == 0:
        return (float("nan"),) * 4
    rng = np.random.default_rng(SEED)
    ia = rng.integers(0, len(sa), size=(B, len(sa)))
    ib = rng.integers(0, len(sb), size=(B, len(sb)))
    r = sa[ia].sum(1) / ca[ia].sum(1) - sb[ib].sum(1) / cb[ib].sum(1)
    lo, hi = np.percentile(r, [2.5, 97.5])
    return (sa.sum() / ca.sum() - sb.sum() / cb.sum(), lo, hi, (1 + (r <= 0).sum()) / (B + 1))

def fmt_b(t):
    return "—" if t[0] != t[0] else f"{t[0]:.2f} [{t[1]:.2f}, {t[2]:.2f}]"

def sel(model, cond, group, qform="ver", data=None):
    data = SC if data is None else data
    d = data[(data.model == model) & (data.condition == cond) & (data.group == group) & (data.qform == qform)]
    return d.sort_values("base_id")

GROUPS = [g for g in ("changed", "label", "stable") if (SC.group == g).any()]
T1 = []
for m in [x["tag"] for x in MODELS]:
    for c, spec in CONDITIONS.items():
        for qf in spec["qforms"]:
            row = {"model": m, "condition": c, "qform": qf}
            if "changed" in GROUPS:
                row["changed: lỗi thời"] = fmt_b(boot_rate(sel(m, c, "changed", qf), "drift_ext"))
                row["changed: acc"] = fmt_b(boot_rate(sel(m, c, "changed", qf), "em"))
            if "label" in GROUPS:
                row["label: tên cũ"] = fmt_b(boot_rate(sel(m, c, "label", qf), "label_drift"))
            if "stable" in GROUPS:
                row["stable: acc"] = fmt_b(boot_rate(sel(m, c, "stable", qf), "em"))
            T1.append(row)
T1 = pd.DataFrame(T1)
T1.to_csv(OUT / "summary_conditions.csv", index=False)
T1

,model,condition,qform,changed: lỗi thời,changed: acc,stable: acc
0,qwen3-8b,new,ver,"0.07 [0.04, 0.12]","0.80 [0.73, 0.86]","1.00 [1.00, 1.00]"
1,qwen3-8b,old,ver,"1.00 [1.00, 1.00]","0.00 [0.00, 0.00]","1.00 [1.00, 1.00]"
2,qwen3-8b,old_new,ver,"0.95 [0.92, 0.98]","0.05 [0.02, 0.08]","1.00 [1.00, 1.00]"
3,qwen3-8b,new_old,ver,"0.97 [0.94, 0.99]","0.03 [0.01, 0.06]","1.00 [1.00, 1.00]"
4,qwen3-8b,tag_old_new,ver,"0.20 [0.14, 0.26]","0.80 [0.73, 0.86]","1.00 [1.00, 1.00]"
...,...,...,...,...,...,...
65,mistral7b-v0.3,vtag_new_old,nover,"0.72 [0.66, 0.78]","0.17 [0.12, 0.22]","1.00 [1.00, 1.00]"
66,mistral7b-v0.3,swap_old_new,ver,"0.98 [0.96, 1.00]","0.02 [0.00, 0.04]","1.00 [1.00, 1.00]"
67,mistral7b-v0.3,swap_new_old,ver,"1.00 [0.98, 1.00]","0.00 [0.00, 0.02]","1.00 [1.00, 1.00]"
68,mistral7b-v0.3,bridge_old_new,ver,"0.04 [0.02, 0.07]","0.94 [0.91, 0.97]","1.00 [1.00, 1.00]"


## 8. Kiểm định khẳng định C1–C6

Mỗi dòng là một (mô hình × giả thuyết). Các cột:

- `p một phía`: giá trị p cho giả thuyết hiệu > 0, từ phép đổi dấu theo cụm;
- `p Holm`: giá trị p sau hiệu chỉnh Holm trong họ kiểm định của mô hình đó;
- `đạt`: ✓ nếu `p Holm` < 0,05.

C1 ở nhánh B không nằm trong họ kiểm định, nên chỉ báo cáo khoảng tin cậy hai phía.

In [47]:
POOL_TAG  = ["tag_old_new", "tag_new_old"]
POOL_NONE = ["old_new", "new_old"]
POOL_VTAG = ["vtag_old_new", "vtag_new_old"]
POOL_SWAP = ["swap_old_new", "swap_new_old"]
POOL_BRDG = ["bridge_old_new", "bridge_new_old"]
not_d4  = lambda d: d[d.template != "D4_id_currency"]
d1_rev  = lambda d: d[(d.template == "D1_tech_tactic") & d.revoked]
HYPS = {   # id: (tên, cột, [(điều kiện, cách hỏi)] phía A, phía B, bộ lọc dữ liệu)
    "C1": ("Có mặt đoạn cũ gây lỗi thời", "drift_ext", [("new_old", "ver")], [("new", "ver")], None),
    "C2": ("Nhãn phiên bản giảm lỗi thời", "drift_ext", [(c, "ver") for c in POOL_NONE], [(c, "ver") for c in POOL_TAG], None),
    "C3": ("Chữ current/superseded cần thiết", "drift_ext", [(c, "ver") for c in POOL_VTAG], [(c, "ver") for c in POOL_TAG], None),
    "C4": ("Nhãn số hoạt động nhờ khớp chuỗi", "drift_ext", [(c, "nover") for c in POOL_VTAG], [(c, "ver") for c in POOL_VTAG], not_d4),
    "C5": ("Siêu dữ liệu lái câu trả lời", "drift_ext", [(c, "ver") for c in POOL_SWAP], [(c, "ver") for c in POOL_TAG], None),
    "C6": ("Cầu nối giúp trả lời đúng (acc)", "em", [(c, "ver") for c in POOL_BRDG], [(c, "ver") for c in POOL_TAG], d1_rev),
}

def side(m, specs, filt):
    data = SC if filt is None else filt(SC)
    return pd.concat([sel(m, c, "changed", qf, data) for c, qf in specs])

def holm(ps):
    order = np.argsort(ps)
    adj, run = np.empty(len(ps)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, (len(ps) - rank) * ps[i]))
        adj[i] = run
    return adj

FLOOR = 0.15                       # quy tắc sàn cho C2 (định trước; ở phân tích độ nhạy vẫn xét bằng drift_ext)
_json_rate = SC.groupby("model")["format"].apply(lambda x: (x == "json").mean()).to_dict()
TECH_FAIL = set(FAILED) | {m["tag"] for m in MODELS if _json_rate.get(m["tag"], 0.0) < JSON_MIN}
for m in TECH_FAIL - set(FAILED):
    FAILED[m] = f"chạy chính: tỉ lệ JSON {_json_rate.get(m, 0.0):.2f} < {JSON_MIN}"
(OUT / "tech_fail.json").write_text(json.dumps(FAILED, indent=2, ensure_ascii=False))   # C7 đọc file này
if TECH_FAIL:
    print("⚠ Mô hình lỗi kỹ thuật (tính là không đạt mọi giả thuyết):", TECH_FAIL)

def confirm_table(strict=False):
    out = []
    for m in [x["tag"] for x in MODELS]:
        rows = []
        untagged = side(m, [(c, "ver") for c in POOL_NONE], None)["drift_ext"].mean()
        for hid, (name, col, A, B, filt) in HYPS.items():
            col = "stale13" if (strict and col == "drift_ext") else col
            a, b = side(m, A, filt), side(m, B, filt)
            d = boot_paired(a, b, col)
            in_family = hid in CFG["family"]
            floor = hid == "C2" and untagged < FLOOR and m not in TECH_FAIL
            rows.append({"model": m, "giả thuyết": hid, "tên": name, "chỉ số": col, "hiệu [KTC 95%]": fmt_b(d),
                         "p một phía": signflip_p(a, b, col) if in_family else float("nan"),
                         "n cụm": a[CLUSTER].nunique(), "trong họ": in_family and not floor and m not in TECH_FAIL,
                         "_floor": floor, "_report": not in_family, "_lo": d[1], "_hi": d[2]})
        fam = [r for r in rows if r["trong họ"]]
        adj = holm(np.array([r["p một phía"] for r in fam])) if fam else []
        for r, x in zip(fam, adj):
            r["p Holm"] = x
            r["đạt"] = "✓" if x < ALPHA else "·"
        for r in rows:
            if not r["trong họ"]:
                r["p Holm"] = float("nan")
                r["đạt"] = ("(báo cáo)" if r["_report"] else "lỗi kỹ thuật" if m in TECH_FAIL
                            else "sàn – không đánh giá được")
            # mô hình lỗi kỹ thuật hoặc ở sàn không được tính là "ngược chiều"
            r["ngược chiều"] = bool(r["_hi"] < 0) and m not in TECH_FAIL and not r["_floor"]
        out += rows
    return pd.DataFrame(out).drop(columns=["_report"])

def verdicts(T):
    res = []
    n_models = len(MODELS)
    for hid in CFG["family"]:
        t = T[T["giả thuyết"] == hid]
        n_floor = int(t["_floor"].sum())
        n_eval = n_models - n_floor
        need = CFG["min_pass"] if n_floor == 0 else math.ceil(0.8 * n_eval)
        n_pass, n_rev = int((t["đạt"] == "✓").sum()), int(t["ngược chiều"].sum())
        if n_eval < 3:
            verdict = "không đánh giá được (sàn)"
        else:
            verdict = "KHẲNG ĐỊNH" if (n_pass >= need and n_rev == 0) else "không khẳng định"
        res.append({"giả thuyết": hid, "tên": HYPS[hid][0], "số mô hình đạt": f"{n_pass}/{n_eval}",
                    "ngưỡng": need, "số mô hình ngược chiều": n_rev, "số mô hình ở sàn": n_floor,
                    "kết luận": verdict})
    return pd.DataFrame(res)

T2 = confirm_table()
T2.drop(columns=["_lo", "_hi", "_floor"]).to_csv(OUT / "confirmatory.csv", index=False)
V2 = verdicts(T2)
V2.to_csv(OUT / "confirmatory_verdicts.csv", index=False)
display(T2.drop(columns=["_lo", "_hi", "_floor"]).round({"p một phía": 4, "p Holm": 4}))
print(f"\nQuy tắc: đạt ở ít nhất {CFG['min_pass']}/{len(MODELS)} mô hình (C2: xem cột 'ngưỡng' nếu có mô hình ở sàn) "
      "và không mô hình nào ngược chiều")
V2

,model,giả thuyết,tên,chỉ số,hiệu [KTC 95%],p một phía,n cụm,trong họ,p Holm,đạt,ngược chiều
0,qwen3-8b,C1,Có mặt đoạn cũ gây lỗi thời,drift_ext,"0.90 [0.85, 0.94]",NaN,133,False,NaN,(báo cáo),False
1,qwen3-8b,C2,Nhãn phiên bản giảm lỗi thời,drift_ext,"0.72 [0.66, 0.77]",0.0001,133,True,0.0005,✓,False
2,qwen3-8b,C3,Chữ current/superseded cần thiết,drift_ext,"0.27 [0.23, 0.31]",0.0001,133,True,0.0005,✓,False
3,qwen3-8b,C4,Nhãn số hoạt động nhờ khớp chuỗi,drift_ext,"0.21 [0.14, 0.28]",0.0001,69,True,0.0005,✓,False
4,qwen3-8b,C5,Siêu dữ liệu lái câu trả lời,drift_ext,"0.74 [0.68, 0.80]",0.0001,133,True,0.0005,✓,False
5,qwen3-8b,C6,Cầu nối giúp trả lời đúng (acc),em,"0.31 [0.20, 0.41]",0.0001,59,True,0.0005,✓,False
6,llama3.1-8b,C1,Có mặt đoạn cũ gây lỗi thời,drift_ext,"0.41 [0.34, 0.49]",NaN,133,False,NaN,(báo cáo),False
7,llama3.1-8b,C2,Nhãn phiên bản giảm lỗi thời,drift_ext,"0.40 [0.34, 0.45]",0.0001,133,True,0.0005,✓,False
8,llama3.1-8b,C3,Chữ current/superseded cần thiết,drift_ext,"0.06 [0.03, 0.09]",0.0003,133,True,0.0006,✓,False
9,llama3.1-8b,C4,Nhãn số hoạt động nhờ khớp chuỗi,drift_ext,"0.25 [0.18, 0.32]",0.0001,69,True,0.0005,✓,False



Quy tắc: đạt ở ít nhất 4/5 mô hình (C2: xem cột 'ngưỡng' nếu có mô hình ở sàn) và không mô hình nào ngược chiều


,giả thuyết,tên,số mô hình đạt,ngưỡng,số mô hình ngược chiều,số mô hình ở sàn,kết luận
0,C2,Nhãn phiên bản giảm lỗi thời,5/5,4,0,0,KHẲNG ĐỊNH
1,C3,Chữ current/superseded cần thiết,5/5,4,0,0,KHẲNG ĐỊNH
2,C4,Nhãn số hoạt động nhờ khớp chuỗi,5/5,4,0,0,KHẲNG ĐỊNH
3,C5,Siêu dữ liệu lái câu trả lời,5/5,4,0,0,KHẲNG ĐỊNH
4,C6,Cầu nối giúp trả lời đúng (acc),3/5,4,0,0,không khẳng định


### 8.1. Phân tích độ nhạy: chỉ số thay thế `stale13`

In [48]:
T2s = confirm_table(strict=True)
T2s.drop(columns=["_lo", "_hi", "_floor"]).to_csv(OUT / "confirmatory_strict.csv", index=False)
V2s = verdicts(T2s)
V2s.to_csv(OUT / "confirmatory_verdicts_strict.csv", index=False)
display(T2s[["model", "giả thuyết", "hiệu [KTC 95%]", "p Holm", "đạt"]].round({"p Holm": 4}))
V2s

,model,giả thuyết,hiệu [KTC 95%],p Holm,đạt
0,qwen3-8b,C1,"0.90 [0.85, 0.94]",NaN,(báo cáo)
1,qwen3-8b,C2,"0.72 [0.66, 0.77]",0.0005,✓
2,qwen3-8b,C3,"0.27 [0.23, 0.31]",0.0005,✓
3,qwen3-8b,C4,"0.21 [0.14, 0.28]",0.0005,✓
4,qwen3-8b,C5,"0.74 [0.68, 0.80]",0.0005,✓
5,qwen3-8b,C6,"0.31 [0.20, 0.41]",0.0005,✓
6,llama3.1-8b,C1,"0.41 [0.34, 0.49]",NaN,(báo cáo)
7,llama3.1-8b,C2,"0.40 [0.34, 0.45]",0.0005,✓
8,llama3.1-8b,C3,"0.06 [0.03, 0.09]",0.0006,✓
9,llama3.1-8b,C4,"0.25 [0.18, 0.32]",0.0005,✓


,giả thuyết,tên,số mô hình đạt,ngưỡng,số mô hình ngược chiều,số mô hình ở sàn,kết luận
0,C2,Nhãn phiên bản giảm lỗi thời,5/5,4,0,0,KHẲNG ĐỊNH
1,C3,Chữ current/superseded cần thiết,5/5,4,0,0,KHẲNG ĐỊNH
2,C4,Nhãn số hoạt động nhờ khớp chuỗi,5/5,4,0,0,KHẲNG ĐỊNH
3,C5,Siêu dữ liệu lái câu trả lời,5/5,4,0,0,KHẲNG ĐỊNH
4,C6,Cầu nối giúp trả lời đúng (acc),3/5,4,0,0,không khẳng định


### 8.2. Báo cáo phụ (không nằm trong họ kiểm định)

Các bảng dưới chỉ để mô tả, không dùng để kết luận khẳng định:

- **Nhánh A**: C2–C5 tính trên nhóm `label`, với chỉ số tên cũ.
- **Cả hai nhánh**: đối chứng `stable`, so độ chính xác mỗi điều kiện với `new`.
- **Cả hai nhánh**: phân bố kết quả v1.3 ở nhóm `changed`.

In [49]:
if "label" in GROUPS:
    LB = []
    for m in [x["tag"] for x in MODELS]:
        for hid in ("C2", "C3", "C4", "C5"):
            name, _, A, B, filt = HYPS[hid]
            data = SC if filt is None else filt(SC)
            a = pd.concat([sel(m, c, "label", qf, data) for c, qf in A])
            b = pd.concat([sel(m, c, "label", qf, data) for c, qf in B])
            LB.append({"model": m, "giả thuyết": hid, "tên": name, "hiệu (tên cũ)": fmt_b(boot_paired(a, b, "label_drift"))})
    display(pd.DataFrame(LB))
ST = []
for m in [x["tag"] for x in MODELS]:
    for c in CONDITIONS:
        if c == "new":
            continue
        ST.append({"model": m, "condition": c,
                   "stable acc − new": fmt_b(boot_paired(sel(m, c, "stable"), sel(m, "new", "stable"), "em", B=2000))})
display(pd.DataFrame(ST).pivot(index="condition", columns="model", values="stable acc − new").reindex(
    [c for c in CONDITIONS if c != "new"]))
CH = SC[(SC.group == "changed") & (SC.qform == "ver")]
T5 = pd.crosstab([CH.model, CH.condition], CH.outcome13, normalize="index").round(3)
T5.to_csv(OUT / "summary_outcomes13.csv")
T5

model,gemma2-9b,llama3.1-8b,mistral7b-v0.3,phi4-mini,qwen3-8b
condition,,,,,
old,"0.01 [0.00, 0.03]","0.00 [-0.02, 0.02]","0.00 [0.00, 0.00]","0.00 [0.00, 0.00]","0.00 [0.00, 0.00]"
old_new,"0.01 [0.00, 0.03]","-0.09 [-0.15, -0.04]","0.02 [0.00, 0.04]","0.01 [0.00, 0.03]","0.00 [0.00, 0.00]"
new_old,"0.01 [0.00, 0.03]","-0.09 [-0.14, -0.04]","0.01 [-0.01, 0.04]","0.01 [0.00, 0.02]","0.00 [0.00, 0.00]"
tag_old_new,"0.00 [-0.02, 0.03]","-0.05 [-0.09, -0.01]","0.02 [0.00, 0.04]","0.01 [0.00, 0.03]","0.00 [0.00, 0.00]"
tag_new_old,"0.01 [0.00, 0.03]","-0.03 [-0.07, 0.01]","0.02 [0.00, 0.04]","0.01 [0.00, 0.03]","0.00 [0.00, 0.00]"
vtag_old_new,"0.00 [-0.03, 0.03]","-0.10 [-0.15, -0.05]","0.02 [0.00, 0.04]","0.01 [0.00, 0.03]","0.00 [0.00, 0.00]"
vtag_new_old,"0.01 [0.00, 0.03]","-0.04 [-0.09, -0.01]","0.02 [0.00, 0.04]","0.01 [-0.01, 0.03]","0.00 [0.00, 0.00]"
swap_old_new,"0.01 [0.00, 0.03]","-0.02 [-0.05, 0.01]","0.02 [0.00, 0.04]","0.01 [0.00, 0.03]","0.00 [0.00, 0.00]"
swap_new_old,"0.01 [0.00, 0.03]","-0.06 [-0.10, -0.01]","0.02 [0.00, 0.04]","0.01 [-0.01, 0.03]","0.00 [0.00, 0.00]"


outcome13                      CORRECT  INCONSISTENT  NO_ANSWER  PARTIAL  \
model          condition                                                   
gemma2-9b      bridge_new_old    0.766         0.000      0.209    0.000   
               bridge_old_new    0.910         0.000      0.085    0.000   
               new               0.851         0.005      0.100    0.020   
               new_old           0.080         0.000      0.000    0.000   
               old               0.000         0.000      0.010    0.000   
               old_new           0.249         0.000      0.005    0.000   
               swap_new_old      0.000         0.000      0.000    0.000   
               swap_old_new      0.000         0.000      0.025    0.000   
               tag_new_old       0.692         0.000      0.000    0.005   
               tag_old_new       0.925         0.000      0.005    0.005   
               vtag_new_old      0.602         0.000      0.020    0.000   
               vtag_old_new      0.816         0.000      0.015    0.005   
llama3.1-8b    bridge_new_old    0.900         0.000      0.000    0.005   
               bridge_old_new    0.950         0.000      0.000    0.005   
               new               0.950         0.000      0.000    0.040   
               new_old           0.418         0.000      0.000    0.139   
               old               0.000         0.000      0.000    0.000   
               old_new           0.353         0.000      0.000    0.040   
               swap_new_old      0.005         0.000      0.000    0.005   
               swap_old_new      0.174         0.000      0.000    0.025   
               tag_new_old       0.791         0.000      0.000    0.035   
               tag_old_new       0.945         0.000      0.000    0.010   
               vtag_new_old      0.692         0.000      0.000    0.040   
               vtag_old_new      0.915         0.000      0.000    0.020   
mistral7b-v0.3 bridge_new_old    0.811         0.000      0.000    0.040   
               bridge_old_new    0.940         0.000      0.000    0.015   
               new               0.771         0.000      0.000    0.080   
               new_old           0.040         0.000      0.000    0.025   
               old               0.010         0.000      0.000    0.020   
               old_new           0.035         0.000      0.000    0.015   
               swap_new_old      0.005         0.000      0.000    0.000   
               swap_old_new      0.020         0.000      0.000    0.000   
               tag_new_old       0.756         0.000      0.000    0.035   
               tag_old_new       0.856         0.000      0.000    0.030   
               vtag_new_old      0.318         0.000      0.000    0.040   
               vtag_old_new      0.552         0.000      0.000    0.020   
phi4-mini      bridge_new_old    0.771         0.010      0.000    0.005   
               bridge_old_new    0.866         0.015      0.000    0.020   
               new               0.547         0.010      0.050    0.005   
               new_old           0.065         0.000      0.000    0.000   
               old               0.000         0.000      0.000    0.000   
               old_new           0.065         0.000      0.000    0.000   
               swap_new_old      0.005         0.000      0.000    0.005   
               swap_old_new      0.040         0.000      0.000    0.000   
               tag_new_old       0.209         0.000      0.000    0.000   
               tag_old_new       0.627         0.000      0.000    0.000   
               vtag_new_old      0.100         0.000      0.000    0.005   
               vtag_old_new      0.289         0.000      0.000    0.000   
qwen3-8b       bridge_new_old    0.891         0.000      0.000    0.000   
               bridge_old_new    0.930         0.000      0.000    0.000   
               new               0.801         0

## 9. C7: so sánh hai lần chuyển (thiên vị kiến thức sẵn có)

Phần này chỉ chạy được khi đã có kết quả của **cả hai** nhánh.

- **Nhánh A:** Qwen và Llama lấy từ notebook 06 (`results/mech-18.1-19.2`); ba mô hình mới lấy từ `results/conf-18.1-19.2`.
- **Nhánh B:** lấy từ `results/conf-6.3-7.2`.

Chỉ số là lỗi thời của ngữ cảnh **không nhãn** (`old_new` và `new_old` gộp), nhóm `changed`, cách hỏi `ver`.

- **Ước lượng phân tầng** theo ba tầng: D1 còn hiệu lực, D1 có mã bị thu hồi, D4 có mã bị thu hồi.
- **Trọng số** của mỗi tầng: w = nA·nB / (nA + nB).
- **Bootstrap** làm độc lập trong từng tầng của từng nhánh. Một cụm dòng dõi có thể nằm ở nhiều tầng (D1 thu hồi và D4 thu hồi của cùng một mã). Thầy đã kiểm tra: lấy mẫu chung theo cụm chỉ làm khoảng tin cậy đổi dưới 0,01.
- **Mô hình lỗi kỹ thuật** ở bất kỳ nhánh nào được tính là không đạt C7.
- **Hiệu chỉnh Holm** trên 5 mô hình.
- **Cấu hình môi trường:** cột "cấu hình A/B" ghi cấu hình của thang dự phòng mà mô hình dùng ở mỗi nhánh. Nếu hai nhánh khác cấu hình, C7 vẫn được kiểm định như thường, và sự khác biệt được nêu trong bài.
- **Dự đoán:** lỗi thời ở v18.1 → v19.2 cao hơn ở v6.3 → v7.2 (hiệu > 0).

In [50]:
def load_scored(p):
    d = pd.read_csv(p)
    for c in ("drift_ext", "em", "label_drift", "revoked", "stale13", "inconsistent"):
        if c in d:
            d[c] = d[c].astype(str).str.strip().str.lower().eq("true")
    if "qform" not in d:
        d["qform"] = "ver"
    return d

SRC_A = {"qwen3-8b": ROOT / "results/mech-18.1-19.2/scored.csv", "llama3.1-8b": ROOT / "results/mech-18.1-19.2/scored.csv"}
for m in ALL_MODELS:
    SRC_A.setdefault(m["tag"], ROOT / "results/conf-18.1-19.2/scored.csv")
SRC_B = ROOT / "results/conf-6.3-7.2/scored.csv"
STRATA = [("D1_tech_tactic", False), ("D1_tech_tactic", True), ("D4_id_currency", True)]

def stratified_diff(a, b, col="drift_ext", B=BOOT_B):
    # Hiệu phân tầng A − B: tầng = (mẫu, có mã bị thu hồi); trọng số w = nA·nB/(nA+nB); bootstrap cụm độc lập trong từng tầng
    rng = np.random.default_rng(SEED + 2)
    est, boot, wsum, parts = 0.0, np.zeros(B), 0.0, []
    for tpl, rev in STRATA:
        sa = a[(a.template == tpl) & (a.revoked == rev)]
        sb = b[(b.template == tpl) & (b.revoked == rev)]
        assert len(sa) and len(sb), f"Thiếu tầng {tpl}/{'thu hồi' if rev else 'còn hiệu lực'} ở một nhánh — gửi thầy"
        w = len(sa) * len(sb) / (len(sa) + len(sb))
        xa, na, _ = _agg(sa, col)
        xb, nb, _ = _agg(sb, col)
        ia = rng.integers(0, len(xa), size=(B, len(xa)))
        ib = rng.integers(0, len(xb), size=(B, len(xb)))
        est += w * (xa.sum() / na.sum() - xb.sum() / nb.sum())
        boot += w * (xa[ia].sum(1) / na[ia].sum(1) - xb[ib].sum(1) / nb[ib].sum(1))
        wsum += w
        parts.append(f"{tpl[:2]}{'-thu hồi' if rev else ''}: {xa.sum() / na.sum():.2f} vs {xb.sum() / nb.sum():.2f} "
                     f"(n {len(sa)}/{len(sb)})")
    if wsum == 0:
        return (float("nan"),) * 4 + ("",)
    est, boot = est / wsum, boot / wsum
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return est, lo, hi, (1 + (boot <= 0).sum()) / (B + 1), "; ".join(parts)

def rungs_of(conf_dir):
    f = conf_dir / "env_rung.json"
    return {t: v["rung"] for t, v in json.loads(f.read_text()).get("rung", {}).items()} if f.exists() else {}

def tech_failed(conf_dir):
    f = conf_dir / "tech_fail.json"
    return set(json.loads(f.read_text())) if f.exists() else set()

dir_A, dir_B = ROOT / "results/conf-18.1-19.2", ROOT / "results/conf-6.3-7.2"
ready = (dir_A / "scored.csv").exists() and SRC_B.exists() and SRC_A["qwen3-8b"].exists()
if ready:
    B_ = load_scored(SRC_B)
    fail_any = tech_failed(dir_A) | tech_failed(dir_B)
    rA, rB = rungs_of(dir_A), rungs_of(dir_B)
    rA.update({"qwen3-8b": 0, "llama3.1-8b": 0})      # nhánh A của Qwen/Llama: notebook 06, cấu hình mặc định
    C7 = []
    for m in [x["tag"] for x in ALL_MODELS]:
        A_ = load_scored(SRC_A[m])
        pick = lambda d: d[(d.model == m) & d.condition.isin(POOL_NONE) & (d.group == "changed") & (d.qform == "ver")]
        a, b = pick(A_), pick(B_)
        cfg = f"{rA.get(m, '?')}/{rB.get(m, '?')}" + (" (khác)" if m in rA and m in rB and rA[m] != rB[m] else "")
        if m in fail_any or len(a) == 0 or len(b) == 0:
            C7.append({"model": m, "hiệu phân tầng [KTC 95%]": "—", "p một phía": float("nan"),
                       "theo tầng (A vs B)": "lỗi kỹ thuật / thiếu dữ liệu (tính là không đạt)",
                       "cấu hình A/B": cfg, "_hi": float("nan")})
            continue
        est, lo, hi, pv, parts = stratified_diff(a, b)
        C7.append({"model": m, "hiệu phân tầng [KTC 95%]": f"{est:.2f} [{lo:.2f}, {hi:.2f}]", "p một phía": pv,
                   "theo tầng (A vs B)": parts, "cấu hình A/B": cfg, "_hi": hi})
    T7 = pd.DataFrame(C7)
    # Holm luôn trên đủ 5 mô hình: mô hình lỗi kỹ thuật được gán p = 1
    T7["p Holm"] = holm(T7["p một phía"].fillna(1.0).to_numpy())
    T7["đạt"] = np.where(T7["p Holm"] < ALPHA, "✓", "·")
    n_pass, n_rev = int((T7["đạt"] == "✓").sum()), int((T7["_hi"] < 0).sum())
    T7 = T7.drop(columns=["_hi"])
    T7.to_csv(ROOT / "results" / "conf_C7_cross_transition.csv", index=False)
    display(T7.round({"p một phía": 4, "p Holm": 4}))
    print(f"C7: đạt {n_pass}/5, ngược chiều {n_rev} → "
          + ("KHẲNG ĐỊNH" if n_pass >= 4 and n_rev == 0 else "không khẳng định"))
else:
    print("Chưa đủ kết quả của cả hai nhánh — chạy lại ô này sau khi chạy nhánh còn lại.")

,model,hiệu phân tầng [KTC 95%],p một phía,theo tầng (A vs B),cấu hình A/B,p Holm,đạt
0,qwen3-8b,"0.05 [0.00, 0.11]",0.0190,D1: 0.83 vs 0.70 (n 144/20); D1-thu hồi: 1.00 ...,0/0,0.0612,·
1,llama3.1-8b,"0.28 [0.19, 0.37]",0.0001,D1: 0.81 vs 0.90 (n 144/20); D1-thu hồi: 1.00 ...,0/0,0.0005,✓
2,gemma2-9b,"0.05 [-0.01, 0.12]",0.0565,D1: 0.76 vs 0.90 (n 144/20); D1-thu hồi: 1.00 ...,0/0,0.1130,·
3,phi4-mini,"0.05 [0.00, 0.11]",0.0153,D1: 0.84 vs 0.90 (n 144/20); D1-thu hồi: 1.00 ...,0/0,0.0612,·
4,mistral7b-v0.3,"-0.01 [-0.07, 0.05]",0.5968,D1: 0.70 vs 0.80 (n 144/20); D1-thu hồi: 0.93 ...,0/0,0.5968,·


C7: đạt 1/5, ngược chiều 0 → không khẳng định


## 10. Ghi manifest và đóng gói gửi thầy

In [51]:
MANIFEST["n_queries"] = len(QUERIES)
MANIFEST["runs"] = {}
for m in MODELS:
    for cond in CONDITIONS:
        mp = RAW / m["tag"] / f"meta_{cond}.json"
        if mp.exists():
            MANIFEST["runs"][f"{m['tag']}/{cond}"] = json.loads(mp.read_text())
MANIFEST["outputs"] = {str(p.relative_to(OUT)): sha256(p) for p in sorted(OUT.rglob("*"))
                       if p.is_file() and p.suffix in (".csv", ".jsonl") and "raw_smoke" not in p.parts
                       and not p.parts[len(OUT.parts)].startswith("prompts")}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2, ensure_ascii=False))
ZIP = OUT / "gui_thay.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p != ZIP and not p.parts[len(OUT.parts)].startswith("prompts_smoke") \
                and not p.parts[len(OUT.parts)].startswith("prompts_mergedsys"):
            z.write(p, p.relative_to(OUT))
    c7 = ROOT / "results" / "conf_C7_cross_transition.csv"
    if c7.exists():
        z.write(c7, c7.name)
print(f"Đã tạo {ZIP.relative_to(ROOT)} ({ZIP.stat().st_size / 1e6:.1f} MB)")

Đã tạo results/conf-6.3-7.2/gui_thay.zip (4.2 MB)


## 11. Gửi thầy

Sau **mỗi** lần chạy, em gửi thầy hai thứ:

- notebook đã chạy;
- file `results/conf-<lần chuyển>/gui_thay.zip`.

Sau lần chạy thứ hai, mục 9 (C7) sẽ có kết quả. Nếu lúc đó ô C7 vẫn báo "chưa đủ", em chạy lại riêng ô đó rồi chạy lại mục 10.